In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly as py
import os
from collections import Counter
from ipyleaflet import Marker, Icon, Map, basemaps, basemap_to_tiles, Circle
import datetime
from datetime import date, timedelta, time, datetime
from geopy.geocoders import Nominatim
from ipywidgets import HTML
from IPython.display import display
from ipyleaflet import Map, Marker, Circle, AwesomeIcon

import warnings
warnings.filterwarnings("ignore")

# Duyurular

In [ ]:
dataDuy = pd.read_csv("Veriler/Duyuru Verileri/Duyurular.csv")
df = dataDuy
dataDuy

In [ ]:
dataTip = pd.read_csv("Veriler/Duyuru Verileri/DuyuruTipleri.csv", encoding= 'unicode_escape')
df2 = dataTip
dataTip.head(20)

In [ ]:
df2.rename(columns = {'duyuruID':'DuyuruTipi'}, inplace = True) # Duyuru tiplerinin isimleri ile eşleştirmek için yapıldı.
merged = pd.merge(dataDuy, df2, how="left", on=["DuyuruTipi"]) # Duyuru tipleri ve duyurular eşleştirildi.
merged = merged.drop(columns = ["MudahaleTarihi", "Destination", "Durumu", "GuncellemeTarihi", "KameraId", "KazaDurumu", "KapaliSerit", "gCoordinate", "DuyuruLokasyon", "KazaTipi", "KazaYeri", "KazaKaraNoktaTipi", "siraNo"])
# Gereksiz sütunlar kaldırıldı.
merged

## Duyurular + Hız Verileri

In [ ]:
isimTarama19 = os.scandir("Veriler/Hız Verileri/2019")
tarama19 = os.scandir("Veriler/Hız Verileri/2019")

isimler19 = []
d = 0

for j in isimTarama19:
  isimler19.append(j.name)

j = 0
segIdDizi19 = []

for belge19 in tarama19:
    j += 1
    segIdDizi19.append(belge19.name.split("_")[0]) # Hız verilerinin dosya isimlerinin ilk kısmı segment ID olduğu için onları bir dizide topluyoruz.
print(len(segIdDizi19))

In [ ]:
# Kaç farklı segId yani konum belirteci olduğunu buluyoruz.
segIdTekli19 = list(dict.fromkeys(segIdDizi19)) # Aynı olanları kaldırdık.
print(len(segIdTekli19))

In [ ]:
for j in range(0, len(segIdTekli19)):
    segIdTekli19[j] = int(segIdTekli19[j])

df_hiz19 = pd.DataFrame(segIdTekli19) # Diziden dataframe'e dönüştürüyoruz.
print(df_hiz19)

In [ ]:
df_hiz19.rename(columns = {0:'vSegId'}, inplace = True)

mergedVhiz19 = pd.merge(merged, df_hiz19, how="inner", on=["vSegId"]) # Hız verilerinin dosya ismindeki segmentId'ler ile duyuru verilerindeki segmentId'lerİ eşleştiriyoruz.
mergedVhiz19 = mergedVhiz19.drop_duplicates() # Tekrarlayan satırları kaldırıyoruz.

second_column19 = mergedVhiz19.pop('duyuruTip')
mergedVhiz19.insert(2, 'duyuruTip', second_column19) # Sütunun yerini değiştiriyoruz.
mergedVhiz19.rename(columns = {'duyuruTip':'DuyuruAdi'}, inplace = True)
mergedVhiz19.sort_values(by='vSegId') # Tabloyu vSegId'ye göre sıralıyoruz.

In [ ]:
mergedVhiz19[mergedVhiz19["vSegDir"].isnull()] # Yön bilgisi olmayan satırları görüntülüyoruz.

In [ ]:
mergedVhiz19 = mergedVhiz19.dropna(subset=['vSegDir']) # SegDir değeri boş olan satırları siliyoruz.
mergedVhiz19['vSegDir'] = mergedVhiz19['vSegDir'].astype(int) # Değerlerin 0 ve 1 olarak görünmesi için tipini int'e çeviriyoruz.
mergedVhiz19

In [ ]:
i = 0
j = 0
hizlar18 = []
hizlar19 = []
gerekliHiz18 = []
gerekliHiz19 = []

mergedVhiz19 = mergedVhiz19[(mergedVhiz19['DuyuruTipi'] == 17) | (mergedVhiz19['DuyuruTipi'] == 34) | (mergedVhiz19['DuyuruTipi'] == 35) | (mergedVhiz19['DuyuruTipi'] == 37) | (mergedVhiz19['DuyuruTipi'] == 38)]
# Yalnızca gerekli satırları aldım.
mergedVhiz19 = mergedVhiz19.reset_index(drop=True)
mergedVhiz19

# Hız Verileri

In [ ]:
j = 0
for j in mergedVhiz19.index:
    segId = mergedVhiz19['vSegId'][j]
    segDir = mergedVhiz19['vSegDir'][j]
    if(isimler19.count("{0}_{1}_2019.csv".format(segId, segDir))):
        hiz19 = pd.read_csv("Veriler/Hız Verileri/2019/{}_{}_2019.csv".format(segId, segDir)) # j satırındaki duyuruyla eşleşen hız dosyasını okuyoruz.
        hizlar19.append(hiz19)
        gerekliHiz19.append("{0}_{1}_2019.csv".format(segId, segDir))
        #print(hiz19)
        j += 1

In [ ]:
HizDos19 = pd.DataFrame(gerekliHiz19)
pd.set_option("display.max_rows", None) # Nerelerde ne kadar duyuru olduğunu görüyoruz.
#print(HizDos19[0].value_counts())

In [ ]:
mergedVhiz19 = mergedVhiz19.sort_values(by='GirisTarihi')
mergedVhiz19 = mergedVhiz19.reset_index(drop=True)
#mergedVhiz19.head()

In [ ]:
# 'GirisTarihi' sütununu datetime türüne dönüştüyoruz.
mergedVhiz19['date_time_giris'] = pd.to_datetime(mergedVhiz19['GirisTarihi'], format='%Y-%m-%d %H:%M:%S')

# Tarih ve saat bilgilerini ayrı sütunlara bölüyoruz.
mergedVhiz19["BaslangicGunu"] = mergedVhiz19['date_time_giris'].dt.date
mergedVhiz19["BaslangicSaati"] = mergedVhiz19['date_time_giris'].dt.time

# Gereksiz sütunları siliyoruz.
mergedVhiz19.drop(['GirisTarihi', 'date_time_giris'], axis=1, inplace=True)

# 'BitisTarihi' sütununu datetime türüne dönüştürüyoruz.
mergedVhiz19['date_time_bitis'] = pd.to_datetime(mergedVhiz19['BitisTarihi'], format='%Y-%m-%d %H:%M:%S')

# Tarih ve saat bilgilerini ayrı sütunlara bölüyoruz.
mergedVhiz19["BitisGunu"] = mergedVhiz19['date_time_bitis'].dt.date
mergedVhiz19["BitisSaati"] = mergedVhiz19['date_time_bitis'].dt.time

# Gereksiz sütunları siliyoruz.
mergedVhiz19.drop(['BitisTarihi', 'date_time_bitis'], axis=1, inplace=True)

# İstenilen sırayı belirtiyoruz.
desired_order = [
    'DuyuruId', 'DuyuruTipi', 'DuyuruAdi',
    'BaslangicGunu', 'BaslangicSaati', 'BitisGunu', 'BitisSaati',
    'XCoor', 'YCoor', 'vSegId', 'vSegDir'
]

# Sırayı uyguluyoruz.
mergedVhiz19 = mergedVhiz19[desired_order]

# BaslangicSaati sütunundaki saat bilgisini stringe dönüştürüyoruz ve noktadan sonrasını siliyoruz.
mergedVhiz19['BaslangicSaati'] = mergedVhiz19['BaslangicSaati'].apply(lambda x: x.strftime('%H:%M:%S'))

#Tekrar saat ver itipine dönüştürüyoruz.
mergedVhiz19['BaslangicSaati'] = pd.to_datetime(mergedVhiz19['BaslangicSaati'], format='%H:%M:%S').dt.time

#mergedVhiz19

# Gelişmiş Grafikler

In [ ]:
def dakikaYuvarla(gun_objesi, saat_objesi):
    # Saat verisini datetime objesine çevir
    saat_objesi = datetime.combine(gun_objesi, saat_objesi)

    # Dakikayı en yakın 5'e yuvarla
    yuvarlanmis_dakika = round(saat_objesi.minute / 5) * 5

    # Yeni dakika değerini datetime objesine ekle
    yeni_dakika = datetime(
        saat_objesi.year,
        saat_objesi.month,
        saat_objesi.day,
        saat_objesi.hour,
        yuvarlanmis_dakika)

    # Sonucu istediğiniz formata dönüştür
    sonuc = yeni_dakika.time()

    return sonuc

# Duyurunun başlangıç ve bitiş tarihlerinin saat aralığını al:
def tarihArasiAlReferansTarih(baslangic_gunu_objesi, baslangic_saati_objesi, bitis_gunu_objesi, bitis_saati_objesi):
    # Tarih ve saat verilerini birleştir.
    baslangic_tarihi_ve_saati = datetime.combine(baslangic_gunu_objesi, baslangic_saati_objesi)
    bitis_tarihi = datetime.combine(bitis_gunu_objesi, bitis_saati_objesi)

    # Tarih ve saat aralığındaki satırları seç
    secilen_df = hizDos[(hizDos['date_time'] >= baslangic_tarihi_ve_saati) & (hizDos['date_time'] <= bitis_tarihi)]

    return secilen_df

# Duyurunun başlangıç ve bitiş tarihlerinin saat aralığını geçmiş ve gelecek haftaların aynı günü için al:
def tarihArasiAlDigerGunler(baslangic_gunu_objesi, baslangic_saati_objesi, satir_sayisi):
    # Tarih ve saat verilerini birleştir.
    baslangic_tarihi_ve_saati = datetime.combine(baslangic_gunu_objesi, baslangic_saati_objesi)

    # Tarih ve saat aralığındaki satırları seç
    secilen_df = hizDos[hizDos['date_time'] >= baslangic_tarihi_ve_saati]
    secilen_df = secilen_df.head(satir_sayisi)

    return secilen_df

In [ ]:
mergedVhiz19

In [ ]:
data = {'DuyuruBaslangicTarihi': [],
        'BaslangicSaati': [],
        'DuyuruBitisTarihi': [],
        'BitisSaati': [],
        'XCoor': [],
        'YCoor': [],
        'SegId': [],
        'SegDir': [],
        'DuyuruZamaniOrtalamaHiz': [],
        'DuyuruDisiOrtalamaHiz': [],
        'EtkiOraniTumYil': [],
        'EtkiOrani3HaftaOnceSonra': [],
        'EtkiOrani5HaftaOnceSonra': []}

df = pd.DataFrame(data)
etki_orani_df = pd.DataFrame()

for i in mergedVhiz19.index:

    # mergedVhiz19.BaslangicGunu[] ile tarihi alıyoruz.
    baslangic_tarihi = mergedVhiz19.BaslangicGunu[i]
    print("Duyuru Baslangic Tarihi:", baslangic_tarihi)
    print("Segment Id:", mergedVhiz19.vSegId[i])
    print("Segment Direction:", mergedVhiz19.vSegDir[i])

    tarihListesi = []

    # Tarihten önceki haftaların o aynı günlerini bulmak için geriye doğru döngü:
    tarih = baslangic_tarihi
    while (tarih - timedelta(days=7)).year == 2019:
        tarih -= timedelta(days=7)
        tarihListesi.append(tarih)

    # Tarihten sonraki tarihleri bulmak için ileriye doğru döngü:
    tarih = baslangic_tarihi
    while tarih.year == 2019:
        tarihListesi.append(tarih)
        tarih += timedelta(days=7)

    gunBas = mergedVhiz19.BaslangicGunu[i]
    saatBas = mergedVhiz19.BaslangicSaati[i]

    gunBit = mergedVhiz19.BitisGunu[i]
    saatBit = mergedVhiz19.BitisSaati[i]

    path = 'Veriler/Hız Verileri/2019/'
    file = str(mergedVhiz19.vSegId[i])
    dire = str(mergedVhiz19.vSegDir[i])
    a = path + file + '_' + dire + '_2019.csv'

    hizDos = pd.read_csv(a)
    hizDos.rename(columns = {'0':'date_time', '1':'speed'}, inplace = True)
    hizDos['date_time'] = pd.to_datetime(hizDos['date_time'], format='%Y-%m-%d %H:%M:%S')

    # Belirli saatlerin arasını df olarak alıyoruz.
    tarihler_arasi_df = tarihArasiAlReferansTarih(gunBas, saatBas, gunBit, saatBit)

    satirSayisi = tarihler_arasi_df.shape[0]

    # "speed" sütunundaki değerlerin ortalamasını alıyoruz.
    ortalama_hiz = tarihler_arasi_df['speed'].mean()

    ## Şimdi yılın diğer günlerine 7 gün arayla bakacağız!

    # Sözlük oluşturuyoruz.
    ortalama_hiz_dict = {}
    uc_hafta_once_sonra_ortalama_hiz_dict = {}
    bes_hafta_once_sonra_ortalama_hiz_dict = {}

    for j in tarihListesi:
        # Duyurunun yapıldığı tarih haftanın hangi gününe denk geliyorsa yıl boyunca o günlerin ilgili saat aralığını alıyoruz.
        gunBas = j
        tarihler_arasi_df_yeni = tarihArasiAlDigerGunler(gunBas, saatBas, satirSayisi)

        satirSayisiYeni = tarihler_arasi_df_yeni.shape[0]

        ortalama_hiz_yeni = tarihler_arasi_df_yeni['speed'].mean()

        # Sözlüğe ekliyoruz.
        ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni

        # Etki oranını tüm yıl için değil de k hafta için almak amacıyla k haftanın değerlerini yeni sözlüklere ekliyoruz.
        gun_farki = abs((gunBas - baslangic_tarihi).days)
        if gun_farki <= 21:
            uc_hafta_once_sonra_ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni
        if gun_farki <= 35:
            bes_hafta_once_sonra_ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni

    # 3 sözlükteki boş verileri de siliyoruz.
    for key, value in list(ortalama_hiz_dict.items()):
        if value == -1.0:
            del ortalama_hiz_dict[key]

    for key, value in list(uc_hafta_once_sonra_ortalama_hiz_dict.items()):
        if value == -1.0:
            del uc_hafta_once_sonra_ortalama_hiz_dict[key]

    for key, value in list(bes_hafta_once_sonra_ortalama_hiz_dict.items()):
        if value == -1.0:
            del bes_hafta_once_sonra_ortalama_hiz_dict[key]

    sirali_sozluk = dict(sorted(ortalama_hiz_dict.items()))

#     # Her bir duyuru için çizgi grafiği oluşturuyoruz.
#     tarihler = list(sirali_sozluk.keys())
#     ortalama_hizlar = list(sirali_sozluk.values())

#     plt.figure(figsize=(10, 6))
#     plt.plot(tarihler, ortalama_hizlar, marker='o', linestyle='-')

#     # Belirli bir noktayı farklı renkte işaretliyoruz.
#     nokta_index = tarihler.index(baslangic_tarihi)
#     plt.annotate(f'({tarihler[nokta_index]})',
#                  xy=(tarihler[nokta_index], ortalama_hizlar[nokta_index]),
#                  xytext=(tarihler[nokta_index], ortalama_hizlar[nokta_index] + 5),
#                  arrowprops=dict(facecolor='red', shrink=0.05),
#                  color='red',
#                  ha='center')

#     plt.title('Tarihler ve Ortalama Hızlar')
#     plt.xlabel('Tarih')
#     plt.ylabel('Ortalama Hız')
#     plt.grid(True)
#     plt.show()

    print("Duyuru Zamanindaki Ortalama Hiz:", ortalama_hiz, "\n")

    for key, value in list(ortalama_hiz_dict.items()):
        if key == baslangic_tarihi:
            del ortalama_hiz_dict[key]

    # Asıl tarihin artık bulunmadığı sözlükteki ortalama hızların ortalamasını alıyoruz.
    duyuru_disi_ortalama_hiz_tum_yil = sum(ortalama_hiz_dict.values()) / len(ortalama_hiz_dict)
    #print("Duyuru Zamani Disindaki Haftalarin, Haftanin O Gunune Ait Ortalama Hizi:", duyuru_disi_ortalama_hiz_tum_yil)

    etki_orani_tum_yil = ((ortalama_hiz - duyuru_disi_ortalama_hiz_tum_yil) / duyuru_disi_ortalama_hiz_tum_yil) * 100
    #print(f"\nTüm Yıl Bazında Etki Oranı: {etki_orani_tum_yil:.2f}%")

    del uc_hafta_once_sonra_ortalama_hiz_dict[baslangic_tarihi]
    uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil = sum(uc_hafta_once_sonra_ortalama_hiz_dict.values()) / len(uc_hafta_once_sonra_ortalama_hiz_dict)

    del bes_hafta_once_sonra_ortalama_hiz_dict[baslangic_tarihi]
    bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil = sum(bes_hafta_once_sonra_ortalama_hiz_dict.values()) / len(bes_hafta_once_sonra_ortalama_hiz_dict)

    etki_orani_3_hafta_once_sonra = ((ortalama_hiz - uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) / uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) * 100
    #print(f"3 Haftalık Etki Oranı: {etki_orani_3_hafta_once_sonra:.2f}%")

    etki_orani_5_hafta_once_sonra = ((ortalama_hiz - bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) / bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) * 100
    #print(f"5 HaftalıkEtki Oranı: {etki_orani_5_hafta_once_sonra:.2f}%\n\n\n")

    yeni_satir = {'DuyuruBaslangicTarihi': baslangic_tarihi,
                  'BaslangicSaati': saatBas,
                  'DuyuruBitisTarihi': gunBit,
                  'BitisSaati': saatBit,
                  'XCoor': mergedVhiz19.XCoor[i],
                  'YCoor': mergedVhiz19.YCoor[i],
                  'SegId': mergedVhiz19.vSegId[i],
                  'SegDir': mergedVhiz19.vSegDir[i],
                  'DuyuruZamaniOrtalamaHiz': ortalama_hiz,
                  'DuyuruDisiOrtalamaHiz': duyuru_disi_ortalama_hiz_tum_yil,
                  'EtkiOraniTumYil': round(etki_orani_tum_yil, 2),
                  'EtkiOrani3HaftaOnceSonra': round(etki_orani_3_hafta_once_sonra, 2),
                  'EtkiOrani5HaftaOnceSonra': round(etki_orani_5_hafta_once_sonra, 2)}

    etki_orani_df = etki_orani_df.append(yeni_satir, ignore_index=True)
    etki_orani_df.dropna(inplace=True)

In [ ]:
duyurular = list(etki_orani_df.index)
etki_oranlari = list(etki_orani_df.EtkiOrani5HaftaOnceSonra)

# Bir figür ve alt eksen oluştuyoruz.
fig, ax = plt.subplots(figsize=(10, 6))

# Çizgi grafiği ekliyoruz.
ax.plot(duyurular, etki_oranlari, marker='o', linestyle='-')

# Grafik özellikleri:
ax.set_title('Yol Bakım Çalışmalarının Bulundukları Bölgenin Ortalama Hızlarına 5 Hafta Bazında Etki Oranları')
ax.set_xlabel('Duyuru No')
ax.set_ylabel('Etki Oranı %')
ax.grid(True)

# Grafiği göster.
plt.show()

# Grafiği dosyaya kaydet.
fig.savefig("Grafikler/Duyurular_ve_5HaftalikEtkiOranlari.png")

etki_orani_df.EtkiOrani5HaftaOnceSonra.mean()

# Aynı grafiğin İngilizce'sini çıkartacağız.

# Bir figür ve alt eksen oluştuyoruz.
fig, ax = plt.subplots(figsize=(10, 6))

# Çizgi grafiği ekliyoruz.
ax.plot(duyurular, etki_oranlari, marker='o', linestyle='-')

# Grafik özellikleri:
ax.set_title('Rates of Impact of Road Maintenance Works on Average Speeds of the Regions on the Basis of Previous and Next 5 Weeks')
ax.set_xlabel('Announcement No')
ax.set_ylabel('Impact Rate %')
ax.grid(True)

# Grafiği göster.
plt.show()

# Grafiği dosyaya kaydet.
fig.savefig("Grafikler/Duyurular_ve_5HaftalikEtkiOranlari_EN.png")

In [ ]:
# Duyuruları haftanın günleri bazında değerlendireceğiz.

weekdays = [[],[],[],[],[],[],[]]

for i in etki_orani_df.index:
    day = etki_orani_df.DuyuruBaslangicTarihi[i].weekday()
    if(day==0):
         weekdays[0].append(etki_orani_df.EtkiOrani5HaftaOnceSonra[i])
    elif(day==1):
         weekdays[1].append(etki_orani_df.EtkiOrani5HaftaOnceSonra[i])
    elif(day==2):
         weekdays[2].append(etki_orani_df.EtkiOrani5HaftaOnceSonra[i])
    elif(day==3):
         weekdays[3].append(etki_orani_df.EtkiOrani5HaftaOnceSonra[i])
    elif(day==4):
         weekdays[4].append(etki_orani_df.EtkiOrani5HaftaOnceSonra[i])
    elif(day==5):
         weekdays[5].append(etki_orani_df.EtkiOrani5HaftaOnceSonra[i])
    elif(day==6):
         weekdays[6].append(etki_orani_df.EtkiOrani5HaftaOnceSonra[i])

weekdaysDict = dict()
for i in range(7):
    weekdaysDict[i] = sum(weekdays[i]) / len(weekdays[i])

# Sözlük verilerini sıralayarak x ve y değerlerini ayırıyoruz.
days = list(weekdaysDict.keys())
values = list(weekdaysDict.values())

# Renk paletini tanımlıyoruz.
colors = sns.color_palette("husl", len(days))

# # Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluştuyoruz.
# sns.set(style="whitegrid")
# plt.figure(figsize=(10, 10))  # Grafik boyutunu genişletiyoruz.
# ax = sns.barplot(x=days, y=values, palette=colors, saturation=0.8)  # Saturation parametresini ekliyoruz.

# # Eksen ve başlık etiketlerini ekliyoruz.
# plt.xlabel('Haftanın Günleri')
# plt.ylabel('Etki Oranı %')
# plt.title('Yol Bakım Çalışmalarının Haftanın Günleri Bazında 5 Haftalık Ortalama Hızlara Etki Oranları')

# # x ekseni etiketlerini güncelliyoruz.
# plt.xticks(days, ['Pzt', 'Sal', 'Çar', 'Per', 'Cum', 'Cmt', 'Paz'])

# Sütunların üzerine değerleri yazdırıyoruz.
#for i, v in enumerate(values):
    #ax.text(i, v + 0.01, f"{len(weekdays[i])} Duyuru\nMax: {np.max(weekdays[i]):.2f}\nMin: {np.min(weekdays[i]):.2f}", ha='center', va='top', color='black')

# Grafiği göster.
#plt.show()

In [ ]:
# Duyurunun, günün hangi bölümlerini kapsadığını buluyoruz.

df = etki_orani_df

# Günü bölümlere ayırıyoruz.
bolumler = {
    (time(6, 0), time(11, 0)): 'Sabah',
    (time(11, 0), time(16, 0)): 'Ogle',
    (time(16, 0), time(21, 0)): 'Aksam',
    (time(21, 0), time(23, 59)): 'Gece',
    (time(0, 0), time(6, 0)): 'Gece'
}

# Bölüm bulma fonksiyonu:
def bul_bolum(baslangic_tarihi, baslangic_saat, bitis_tarihi, bitis_saat):
    baslangic_zaman = datetime.combine(baslangic_tarihi, baslangic_saat)
    bitis_zaman = datetime.combine(bitis_tarihi, bitis_saat)

    # Başlangıç ve bitiş arasındaki süreyi kontrol ediyoruz.
    fark = bitis_zaman - baslangic_zaman
    if fark.days >= 1:
        return list(bolumler.values())  # Eğer bir günden fazla süre varsa, tüm bölümleri al.
    else:
        bulunan_bolumler = set()
        for baslangic, bitis in bolumler.keys():
            if baslangic_zaman.time() < bitis:
                if baslangic < bitis_zaman.time() < bitis:
                    bulunan_bolumler.add(bolumler[(baslangic, bitis)])
                elif baslangic < (baslangic_zaman + timedelta(days=1)).time() < bitis:
                    bulunan_bolumler.add(bolumler[(baslangic, bitis)])
        return list(bulunan_bolumler) if bulunan_bolumler else None

# Bütün duyurular için günün hangi bölümlerini kapsadığını buluyoruz.
df['Bolum'] = df.apply(lambda row: bul_bolum(row['DuyuruBaslangicTarihi'], row['BaslangicSaati'], row['DuyuruBitisTarihi'], row['BitisSaati']), axis=1)

df['Bolum'] = df['Bolum'].apply(lambda x: ['Sabah', 'Ogle', 'Aksam', 'Gece'] if 'Gece' in x and x.count("Gece") > 1 else x)

etki_orani_ve_günün_bolumu_df = df

In [ ]:
partOfDay = [[],[],[],[]]

for i in etki_orani_ve_günün_bolumu_df.index:
    part = etki_orani_ve_günün_bolumu_df.Bolum

    for j in part[i]:
        if(j=="Sabah"):
            partOfDay[0].append(etki_orani_ve_günün_bolumu_df.EtkiOrani5HaftaOnceSonra[i])
        if(j=="Ogle"):
            partOfDay[1].append(etki_orani_ve_günün_bolumu_df.EtkiOrani5HaftaOnceSonra[i])
        if(j=="Aksam"):
            partOfDay[2].append(etki_orani_ve_günün_bolumu_df.EtkiOrani5HaftaOnceSonra[i])
        if(j=="Gece"):
            partOfDay[3].append(etki_orani_ve_günün_bolumu_df.EtkiOrani5HaftaOnceSonra[i])

partOfDayDict = dict()
for i in range(4):
    partOfDayDict[i] = sum(partOfDay[i]) / len(partOfDay[i])

# Sözlük verilerini sıralayarak x ve y değerlerini ayırıyoruz.
parts = list(partOfDayDict.keys())
values = list(partOfDayDict.values())

# Renk paletini tanımlayın
colors = sns.color_palette("husl", len(parts))

# Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluşturuyoruz.
# sns.set(style="whitegrid")
# plt.figure(figsize=(10, 6))
# ax = sns.barplot(x=parts, y=values, palette=colors)

# # Eksen ve başlık etiketlerini ekliyoruz.
# plt.xlabel('Günün Bölümleri')
# plt.ylabel('Etki Oranı %')
# plt.title('Yol Bakım Çalışmalarının Günün Bölümleri Bazında 5 Haftalık Ortalama Hızlara Etki Oranları-Duyurunun Tamamı Alınıyor')

# # x ekseni etiketlerini güncelliyoruz.
# plt.xticks(parts, ['06:00-11:00 (Sabah)', '11:00-16:00 (Öğle)', '16:00-21:00 (Akşam)', '21:00-06:00 (Gece)'])

# # Sütunların üzerine değerleri yazdırıyoruz.
# for i, v in enumerate(values):
#     ax.text(i, v + 0.01, f"{len(partOfDay[i])} Duyuru\nMax: {np.max(partOfDay[i]):.2f}\nMin: {np.min(partOfDay[i]):.2f}", ha='center', va='bottom', color='white')

# # Grafiği göster.
# plt.show()

In [ ]:
partOfDayforSpeed = [[],[],[],[]]

for i in etki_orani_ve_günün_bolumu_df.index:
    part = etki_orani_ve_günün_bolumu_df.Bolum

    for j in part[i]:
        if(j=="Sabah"):
            partOfDayforSpeed[0].append(etki_orani_ve_günün_bolumu_df.DuyuruDisiOrtalamaHiz[i])
        elif(j=="Ogle"):
            partOfDayforSpeed[1].append(etki_orani_ve_günün_bolumu_df.DuyuruDisiOrtalamaHiz[i])
        elif(j=="Aksam"):
            partOfDayforSpeed[2].append(etki_orani_ve_günün_bolumu_df.DuyuruDisiOrtalamaHiz[i])
        elif(j=="Gece"):
            partOfDayforSpeed[3].append(etki_orani_ve_günün_bolumu_df.DuyuruDisiOrtalamaHiz[i])

partOfDayforSpeedDict = dict()
for i in range(4):
    partOfDayforSpeedDict[i] = sum(partOfDayforSpeed[i]) / len(partOfDayforSpeed[i])
    print(partOfDayforSpeedDict[i])

# Sözlük verilerini sıralayarak x ve y değerlerini ayırıyoruz.
parts = list(partOfDayforSpeedDict.keys())
values = list(partOfDayforSpeedDict.values())

# Renk paletini tanımlıyoruz.
colors = sns.color_palette("husl", len(parts))

# # Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluşturuyoruz.
# sns.set(style="whitegrid")
# plt.figure(figsize=(10, 6))
# ax = sns.barplot(x=parts, y=values, palette=colors)

# # Eksen ve başlık etiketlerini ekliyoruz.
# plt.xlabel('Günün Bölümleri')
# plt.ylabel('Ortalama Hız')
# plt.title('Günün Bölümleri Bazında Ortalama Hızlar-Duyurunun Tamamı Alınıyor')

# # x ekseni etiketlerini güncelliyoruz.
# plt.xticks(parts, ['06:00-11:00 (Sabah)', '11:00-16:00 (Öğle)', '16:00-21:00 (Akşam)', '21:00-06:00 (Gece)'])

# # Sütunların üzerine değerleri yazdırıyoruz.
# for i, v in enumerate(values):
#     ax.text(i, v + 0.01, f"{len(partOfDayforSpeed[i])} Duyuru\nMax: {np.max(partOfDayforSpeed[i]):.2f}\nMin: {np.min(partOfDayforSpeed[i]):.2f}", ha='center', va='bottom', color='black')

# # Grafiği göster.
# plt.show()

In [ ]:
# Duyuruların hangi ilçeler için yapıldığını bulacağız.

ilceListesi = []

def koordinat_ilce_bul(latitude, longitude):
    geolocator = Nominatim(user_agent="my_geocoder")
    location = geolocator.reverse((latitude, longitude), language="tr")

    if location:
        address = location.address
        ilce = address.split(",")[-5]  # İlçe bilgisini alıyoruz.
        return ilce.strip()
    else:
        return "Bilgi bulunamadı."

# Yeni bir sütun ekleyerek her satıra ilçe bilgisini ekliyoruz.
etki_orani_df['Ilce'] = ""

for i in etki_orani_df.index:
    latitude = etki_orani_df.XCoor[i]
    longitude = etki_orani_df.YCoor[i]

    ilce = koordinat_ilce_bul(latitude, longitude)

    # Her satırın 'Ilce' sütununa ilçe bilgisini ekliyoruz.
    ilceListesi.append(ilce)
    etki_orani_df.at[i, 'Ilce'] = ilce

In [ ]:
ilceler = etki_orani_df['Ilce'].unique()  # Veri içerisindeki unique ilçe isimlerini alıyoruz.

ilce_dict = {}  # Boş bir sözlük oluşturuyoruz.

# Her bir ilçe için ilgili satırları seçip etki oranlarını bir liste olarak sözlüğe ekliyoruz.
for ilce in ilceler:
    ilce_satirlari = etki_orani_df[etki_orani_df['Ilce'] == ilce]
    etki_oranlari = ilce_satirlari['EtkiOrani5HaftaOnceSonra'].tolist()
    ilce_dict[ilce] = etki_oranlari

# Oluşturulan sözlüğü yazdırıyoruz.
#print(ilce_dict, "\n")

# Her bir ilçenin etki oranlarının ortalamasını alıyoruz.
ilce_ortalama = {ilce: sum(etki_oranlari) / len(etki_oranlari) for ilce, etki_oranlari in ilce_dict.items()}

# Oluşturulan sözlüğü yazdırıyoruz.
#print(ilce_ortalama)

# Sözlüğü tuple listesine dönüştürüyoruz.
sorted_ilce_ortalama = sorted(ilce_ortalama.items(), key=lambda x: x[1])

# Tuple listesinden sıralanmış sözlüğü oluşturuyoruz.
ilce_ortalama_sorted = dict(sorted_ilce_ortalama)

ilceNames = list(ilce_ortalama_sorted.keys())
values = list(ilce_ortalama_sorted.values())

# Renk paletini tanımlayın
colors = sns.color_palette("husl", len(parts))

# Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluşturuyoruz.
sns.set(style="whitegrid")
plt.figure(figsize=(15, 8))
ax = sns.barplot(x=ilceNames, y=values, palette=colors)

# Eksen ve başlık etiketlerini ekliyoruz.
plt.xlabel('İlçeler')
plt.ylabel('Etki Oranı %')
plt.title('İlçe Bazında Yol Bakım Çalışmalarının 5 Haftalık Etki Oranları')

# Kılavuz çizgileri ekliyoruz.
for i in range(len(ilceNames)):
    plt.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8)

    # Sütunların üstüne değerleri yazdırıyoruz.
    plt.text(i, values[i] + 0.01, f"{len(ilce_dict[ilceNames[i]])}", ha='center', va='bottom', color='black')

# X ekseni etiketlerini dik yapıyoruz.
plt.xticks(rotation=90, ha='right')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/İlceBazinda_5HaftalikEtkiOranlari")

# Aynı grafiğin İngilizce'sini çıkartacağız.

# Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluşturuyoruz.
sns.set(style="whitegrid")
plt.figure(figsize=(15, 8))
ax = sns.barplot(x=ilceNames, y=values, palette=colors)

# Eksen ve başlık etiketlerini ekliyoruz.
plt.xlabel('District')
plt.ylabel('Impact Rate %')
plt.title('5-Week Impact Rates of Road Maintenance Works by District')

# Kılavuz çizgileri ekliyoruz.
for i in range(len(ilceNames)):
    plt.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8)

    # Sütunların üstüne değerleri yazdırıyoruz.
    plt.text(i, values[i] + 0.01, f"{len(ilce_dict[ilceNames[i]])}", ha='center', va='bottom', color='black')

# X ekseni etiketlerini dik yapıyoruz.
plt.xticks(rotation=90, ha='right')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/İlceBazinda_5HaftalikEtkiOranlari_EN")

In [ ]:
# Duyuruların neresi için yapıldığını bulacağız.

yolAdresiListesi = []

def koordinat_ilce_bul(latitude, longitude):
    geolocator = Nominatim(user_agent="my_geocoder")
    location = geolocator.reverse((latitude, longitude), language="tr")

    if location:
        address = location.address
        yer = address.split(",")[0]  # İlçe bilgisini alıyoruz.
        return yer.strip()
    else:
        return "Bilgi bulunamadı."

for i in etki_orani_df.index:
    latitude = etki_orani_df.XCoor[i]
    longitude = etki_orani_df.YCoor[i]

    yer = koordinat_ilce_bul(latitude, longitude)

    yolAdresiListesi.append(yer)
    etki_orani_df.at[i, 'YolAdresi'] = yer

yol_adresi_counts = etki_orani_df['YolAdresi'].value_counts().to_dict()

#print(yol_adresi_counts)

# YolAdresi sütununu güncelliyoruz.
etki_orani_df['YolAdresi'] = etki_orani_df.apply(lambda row: row['YolAdresi'] + (", Dir:0" if row['SegDir'] == 0.0 else ", Dir:1"), axis=1)

yolAdresleri = etki_orani_df['YolAdresi'].unique()  # Veri içerisindeki unique ilçe isimlerini alıyoruz.

In [ ]:
yol_adresi_dict = {}  # Boş bir sözlük oluşturuyoruz.

# Her bir ilçe için ilgili satırları seçin ve etki oranlarını bir liste olarak sözlüğe ekliyoruz.
for yol in yolAdresleri:
    yol_satirlari = etki_orani_df[etki_orani_df['YolAdresi'] == yol]
    etki_oranlari = yol_satirlari['EtkiOrani5HaftaOnceSonra'].tolist()
    yol_adresi_dict[yol] = etki_oranlari

# Her bir ilçenin etki oranlarının ortalamasını alıyoruz.
yol_adresi_ortalama = {yol: sum(etki_oranlari) / len(etki_oranlari) for yol, etki_oranlari in yol_adresi_dict.items()}

# Oluşturulan sözlüğü yazdırıyoruz.
#print(yol_adresi_ortalama)

# Sözlüğü tuple listesine dönüştürüyoruz.
sorted_yol_ortalama = sorted(yol_adresi_ortalama.items(), key=lambda x: x[1])

# Tuple listesinden sıralanmış sözlüğü oluşturuyoruz.
yol_ortalama_sorted = dict(sorted_yol_ortalama)

yolNames = list(yol_ortalama_sorted.keys())
values = list(yol_ortalama_sorted.values())

# Renk paletini tanımlayın
colors = sns.color_palette("husl", len(parts))

# Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluşturuyoruz.
sns.set(style="whitegrid")
plt.figure(figsize=(15, 8))
ax = sns.barplot(x=yolNames, y=values, palette=colors)

# Eksen ve başlık etiketlerini ekliyoruz.
plt.xlabel('Yollar')
plt.ylabel('Etki Oranı %')
plt.title('Yol ve Yön Bazında Yol Bakım Çalışmalarının 5 Haftalık Etki Oranları')

for i in range(len(yolNames)):
    plt.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8)

    plt.text(i, values[i] + 0.01, f"{len(yol_adresi_dict[yolNames[i]])}", ha='center', va='bottom', color='black')

plt.xticks(rotation=90, ha='right')

#plt.grid(axis='both')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/YolveYönBazinda_5HaftalikEtkiOranlari")

# Aynı grafiğin İngilizce'sini çıkartacağız.

# Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluşturuyoruz.
sns.set(style="whitegrid")
plt.figure(figsize=(15, 8))
ax = sns.barplot(x=yolNames, y=values, palette=colors)

# Eksen ve başlık etiketlerini ekliyoruz.
plt.xlabel('Roadways')
plt.ylabel('Impact Rate %')
plt.title('5-Week Impact Rates of Road Maintenance Works by Road and Direction')

for i in range(len(yolNames)):
    plt.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8)

    plt.text(i, values[i] + 0.01, f"{len(yol_adresi_dict[yolNames[i]])}", ha='center', va='bottom', color='black')

plt.xticks(rotation=90, ha='right')

#plt.grid(axis='both')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/YolveYönBazinda_5HaftalikEtkiOranlari_EN")

In [ ]:
# Df'ye ilçe ve yol adresi bilgileri eklendi.
etki_orani_df

## Duyuruları Günün Bölümlerine Göre Parçalara Ayırıyoruz:

In [ ]:
# Günün bölümleri için belirlenen saat sınırlarının duyuru içerisinde geçip geçmediğinin kontrolü:
def check_time(row):
    target_hours = [6, 11, 16, 21, 0]
    return row.hour in target_hours and row.minute is 0

# Günün bölümünü bulma fonksiyonu:
def bul_bolum(baslangic_tarihi, baslangic_saat, bitis_tarihi, bitis_saat):
    baslangic_zaman = datetime.combine(baslangic_tarihi, baslangic_saat)
    bitis_zaman = datetime.combine(bitis_tarihi, bitis_saat)

    # Başlangıç ve bitiş arasındaki süreyi kontrol ediyoruz.
    fark = bitis_zaman - baslangic_zaman
    if fark.days >= 1:
        return list(bolumler.values())  # Eğer bir günden fazla süre varsa, tüm bölümleri al.
    else:
        bulunan_bolumler = set()
        for baslangic, bitis in bolumler.keys():
            if baslangic_zaman.time() < bitis:
                if baslangic < bitis_zaman.time() < bitis:
                    bulunan_bolumler.add(bolumler[(baslangic, bitis)])
                elif baslangic < (baslangic_zaman + timedelta(days=1)).time() < bitis:
                    bulunan_bolumler.add(bolumler[(baslangic, bitis)])
        return list(bulunan_bolumler) if bulunan_bolumler else None

def dfYeniSatirEkle(i, baslangicTarihi, baslangicSaati, bitisTarihi, bitisSaati, ortalamaHiz):
    yeni_satir = {'DuyuruIndex': mergedVhiz19.index[i],
                  'BaslangicTarihi': baslangicTarihi,
                  'BaslangicSaati': baslangicSaati,
                  'BitisTarihi': bitisTarihi,
                  'BitisSaati': bitisSaati,
                  'XCoor': mergedVhiz19.XCoor[i],
                  'YCoor': mergedVhiz19.YCoor[i],
                  'SegId': mergedVhiz19.vSegId[i],
                  'SegDir': mergedVhiz19.vSegDir[i],
                  'DuyuruZamaniOrtalamaHiz': ortalamaHiz,
                  'GununBolumu': bul_bolum(baslangicTarihi, baslangicSaati, bitisTarihi, bitisSaati)}

    return yeni_satir

In [ ]:
# Duyuruları, günün bölümlerine göre bölerek duyuruları parçalıyoruz.

data = {'DuyuruIndex': [],
        'DuyuruBaslangicTarihi': [],
        'BaslangicSaati': [],
        'DuyuruBitisTarihi': [],
        'BitisSaati': [],
        'XCoor': [],
        'YCoor': [],
        'SegId': [],
        'SegDir': [],
        'DuyuruZamaniOrtalamaHiz': [],
        'GununBolumu': []}

df = pd.DataFrame(data)
bolumler_ayrilmis_etki_orani_df = pd.DataFrame()

bolumler = {
    (time(6, 0), time(11, 0)): 'Sabah',
    (time(11, 0), time(16, 0)): 'Ogle',
    (time(16, 0), time(21, 0)): 'Aksam',
    (time(21, 0), time(23, 59)): 'Gece',
    (time(0, 0), time(6, 0)): 'Gece'}

for i in mergedVhiz19.index:

    # mergedVhiz19.BaslangicGunu[] ile tarihi alıyoruz.
    baslangic_tarihi = mergedVhiz19.BaslangicGunu[i]
    baslangic_saati = mergedVhiz19.BaslangicSaati[i]
    bitis_tarihi = mergedVhiz19.BitisGunu[i]
    bitis_saati = mergedVhiz19.BitisSaati[i]
    print("Duyuru Baslangic Tarihi:", baslangic_tarihi)
    print("Duyuru Baslangic Saati:", baslangic_saati)
    print("Duyuru Bitis Tarihi:", bitis_tarihi)
    print("Duyuru Bitis Saati:", bitis_saati)

    tarihListesi = []

    # Tarihten önceki haftaların o aynı günlerini bulmak için geriye doğru döngü:
    tarih = baslangic_tarihi
    while (tarih - timedelta(days=7)).year == 2019:
        tarih -= timedelta(days=7)
        tarihListesi.append(tarih)

    # Tarihten sonraki tarihleri bulmak için ileriye doğru döngü:
    tarih = baslangic_tarihi
    while tarih.year == 2019:
        tarihListesi.append(tarih)
        tarih += timedelta(days=7)

    gunBas = mergedVhiz19.BaslangicGunu[i]
    saatBas = mergedVhiz19.BaslangicSaati[i]
    #baslangicSaati = dakikaYuvarla(gunBas, saatBas)
    #print(baslangicSaati)

    gunBit = mergedVhiz19.BitisGunu[i]
    saatBit = mergedVhiz19.BitisSaati[i]
    #bitisSaati = dakikaYuvarla(gunBit, saatBit)
    #print(bitisSaati)

    # Duyuruya ait hız verilerinin olduğu dosyayı buluyoruz.
    path = 'Veriler/Hız Verileri/2019/'
    file = str(mergedVhiz19.vSegId[i])
    dire = str(mergedVhiz19.vSegDir[i])
    a = path + file + '_' + dire + '_2019.csv'

    hizDos = pd.read_csv(a)
    hizDos.rename(columns = {'0':'date_time', '1':'speed'}, inplace = True)
    hizDos['date_time'] = pd.to_datetime(hizDos['date_time'], format='%Y-%m-%d %H:%M:%S')
    hizDos = hizDos[hizDos['speed'] != -1.0]

    # Belirli saatlerin arasını df olarak alıyoruz.
    tarihler_arasi_df = tarihArasiAlReferansTarih(gunBas, saatBas, gunBit, saatBit)

    satirSayisi = tarihler_arasi_df.shape[0]

    # "speed" sütunundaki değerlerin ortalamasını alıyoruz.
    ortalama_hiz = tarihler_arasi_df['speed'].mean()
    print("Duyuru Zamanindaki Ortalama Hiz:", ortalama_hiz)

    tarihler_arasi_df['is_target_time'] = tarihler_arasi_df['date_time'].apply(check_time)
    true_rows = tarihler_arasi_df[tarihler_arasi_df['is_target_time'] == True]
    true_indexes = true_rows.index.tolist()

    if(true_indexes): # Eğer duyuru tarihinin içinde günün bölümlerine ait sınırlardan en az biri varsa:
        sayac = 0
        df1 = tarihler_arasi_df.loc[:true_indexes[0]]
        df2 = tarihler_arasi_df.loc[true_indexes[0]+1:]
        last_row_date1 = df1.loc[df1.index[-1], 'date_time'].date()
        last_row_time1 = df1.loc[df1.index[-1], 'date_time'].time()
        ortalama_hiz1 = df1['speed'].mean()

        # İkiye bölünmüş df'nin ilk parçası:
        yeni_satir = dfYeniSatirEkle(i, baslangic_tarihi, baslangic_saati, last_row_date1, last_row_time1, ortalama_hiz1)
        bolumler_ayrilmis_etki_orani_df = bolumler_ayrilmis_etki_orani_df.append(yeni_satir, ignore_index=True)
        sayac += 1
        print("İlk df parcasi")

        k = 0
        while(df2.is_target_time.any()): # 2. df'nin içinde sınır saatleri varsa bunlar bitene kadar çalış.
            true_rows2 = df2[df2['is_target_time'] == True]
            true_indexes2 = true_rows2.index.tolist()
            df1 = df2.loc[:true_indexes2[k]]
            df2 = df2.loc[true_indexes2[k]+1:]

            first_row_date2 = df1.loc[df1.index[0], 'date_time'].date()
            first_row_time2 = df1.loc[df1.index[0], 'date_time'].time()
            last_row_date2 = df1.loc[df1.index[-1], 'date_time'].date()
            last_row_time2 = df1.loc[df1.index[-1], 'date_time'].time()
            ortalama_hiz2 = df1['speed'].mean()

            # Bölünmüş yeni df'nin 1. kısmı:
            yeni_satir = dfYeniSatirEkle(i, first_row_date2, first_row_time2, last_row_date2, last_row_time2, ortalama_hiz2)
            bolumler_ayrilmis_etki_orani_df = bolumler_ayrilmis_etki_orani_df.append(yeni_satir, ignore_index=True)
            sayac += 1
            print("{}. df parcasi".format(sayac))

        if not (df2.empty):
            first_row_date3 = df2.loc[df2.index[0], 'date_time'].date()
            first_row_time3 = df2.loc[df2.index[0], 'date_time'].time()
            last_row_date3 = df2.loc[df2.index[-1], 'date_time'].date()
            last_row_time3 = df2.loc[df2.index[-1], 'date_time'].time()
            ortalama_hiz3 = df2['speed'].mean()

            # Bölünmüş yeni df'nin 2. kısmı (Duyurunun son df'si):
            yeni_satir = dfYeniSatirEkle(i, first_row_date3, first_row_time3, bitis_tarihi, bitis_saati, ortalama_hiz3)
            bolumler_ayrilmis_etki_orani_df = bolumler_ayrilmis_etki_orani_df.append(yeni_satir, ignore_index=True)
            sayac += 1
            print("Son df parcasi")
            print("\n")

    else: # Eğer duyuru saatlerinin içinde sınır saatler yoksa (Duyurunun tek df'si):
        yeni_satir = dfYeniSatirEkle(i, baslangic_tarihi, baslangic_saati, bitis_tarihi, bitis_saati, ortalama_hiz)
        bolumler_ayrilmis_etki_orani_df = bolumler_ayrilmis_etki_orani_df.append(yeni_satir, ignore_index=True)
        print("Tek df parcasi")
        print("\n")

In [ ]:
# Günün bölümlerine göre parçalanmış duyurular:
bolumler_ayrilmis_etki_orani_df['GununBolumu'] = bolumler_ayrilmis_etki_orani_df['GununBolumu'].apply(lambda x: str(x[0]))
bolumler_ayrilmis_etki_orani_df

In [ ]:
# 6 ay süren bir duyuru var ve sonuçları yanlış etkilediğini düşündüğümüz için bu duyuruyu kaldırarak aynı işlemleri tekrarlıyoruz.
bolumler_ayrilmis_etki_orani_df = bolumler_ayrilmis_etki_orani_df[bolumler_ayrilmis_etki_orani_df['DuyuruIndex'] != 33.0]

In [ ]:
# Parçalanmış duyuruların önceki ve sonraki haftalarının hızlarına bakarak etki oranlarını df'e ekliyoruz.

for i in bolumler_ayrilmis_etki_orani_df.index:

    baslangic_tarihi = bolumler_ayrilmis_etki_orani_df.BaslangicTarihi[i]
    print("Duyuru Index:", bolumler_ayrilmis_etki_orani_df.DuyuruIndex[i])

    tarihListesi = []

    # Tarihten önceki haftaların o aynı günlerini bulmak için geriye doğru döngü
    tarih = baslangic_tarihi
    while (tarih - timedelta(days=7)).year == 2019:
        tarih -= timedelta(days=7)
        tarihListesi.append(tarih)

    # Tarihten sonraki tarihleri bulmak için ileriye doğru döngü
    tarih = baslangic_tarihi
    while tarih.year == 2019:
        tarihListesi.append(tarih)
        tarih += timedelta(days=7)

    gunBas = bolumler_ayrilmis_etki_orani_df.BaslangicTarihi[i]
    saatBas = bolumler_ayrilmis_etki_orani_df.BaslangicSaati[i]

    gunBit = bolumler_ayrilmis_etki_orani_df.BitisTarihi[i]
    saatBit = bolumler_ayrilmis_etki_orani_df.BitisSaati[i]

    bolumler_ayrilmis_etki_orani_df['SegId'] = bolumler_ayrilmis_etki_orani_df['SegId'].round().astype(int)
    bolumler_ayrilmis_etki_orani_df['SegDir'] = bolumler_ayrilmis_etki_orani_df['SegDir'].round().astype(int)

    path = 'Veriler/Hız Verileri/2019/'
    file = str(bolumler_ayrilmis_etki_orani_df.SegId[i])
    dire = str(bolumler_ayrilmis_etki_orani_df.SegDir[i])
    a = path + file + '_' + dire + '_2019.csv'

    hizDos = pd.read_csv(a)
    hizDos.rename(columns = {'0':'date_time', '1':'speed'}, inplace = True)
    hizDos['date_time'] = pd.to_datetime(hizDos['date_time'], format='%Y-%m-%d %H:%M:%S')

    # Belirli saatlerin arasını df olarak al.
    tarihler_arasi_df = tarihArasiAlReferansTarih(gunBas, saatBas, gunBit, saatBit)
    #print(tarihler_arasi_df, "\n")

    satirSayisi = tarihler_arasi_df.shape[0]
    #print("Satir Sayisi:", satirSayisi)

    # "speed" sütunundaki değerlerin ortalamasını al.
    ortalama_hiz = tarihler_arasi_df['speed'].mean()

    ## Şimdi yılın diğer günlerine 7 gün arayla bakacağız.

    # Sözlük oluştur
    ortalama_hiz_dict = {}
    uc_hafta_once_sonra_ortalama_hiz_dict = {}
    bes_hafta_once_sonra_ortalama_hiz_dict = {}

    for j in tarihListesi:
        # Duyurunun yapıldığı tarih haftanın hangi gününe denk geliyorsa yıl boyunca o günlerin ilgili saat aralığını al.
        gunBas = j
        tarihler_arasi_df_yeni = tarihArasiAlDigerGunler(gunBas, saatBas, satirSayisi)

        satirSayisiYeni = tarihler_arasi_df_yeni.shape[0]

        ortalama_hiz_yeni = tarihler_arasi_df_yeni['speed'].mean()
        #print("Ortalama Hiz:", ortalama_hiz, "\n")

        # Sözlüğe ekle
        ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni

        # Etki oranını tüm yıl değil de k hafta almak için k haftanın değerlerini yeni sözlüklere de ekliyoruz.
        gun_farki = abs((gunBas - baslangic_tarihi).days)
        if gun_farki <= 21:
            uc_hafta_once_sonra_ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni
        if gun_farki <= 35:
            bes_hafta_once_sonra_ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni

    # 3 sözlükteki boş verileri de siliyoruz.
    for key, value in list(ortalama_hiz_dict.items()):
        if value == -1.0:
            del ortalama_hiz_dict[key]

    for key, value in list(uc_hafta_once_sonra_ortalama_hiz_dict.items()):
        if value == -1.0:
            del uc_hafta_once_sonra_ortalama_hiz_dict[key]

    for key, value in list(bes_hafta_once_sonra_ortalama_hiz_dict.items()):
        if value == -1.0:
            del bes_hafta_once_sonra_ortalama_hiz_dict[key]

    sirali_sozluk = dict(sorted(ortalama_hiz_dict.items()))

    print("Duyuru Zamanindaki Ortalama Hiz:", ortalama_hiz, "\n")

    if (ortalama_hiz == -1.0):
        continue

    for key, value in list(ortalama_hiz_dict.items()):
        if key == baslangic_tarihi:
            del ortalama_hiz_dict[key]

    # Asıl tarihin artık bulunmadığı sözlükteki ortalama hızların ortalamasını al:
    duyuru_disi_ortalama_hiz_tum_yil = sum(ortalama_hiz_dict.values()) / len(ortalama_hiz_dict)
    print("Duyuru Zamani Disindaki Haftalarin, Haftanin O Gunune Ait Ortalama Hizi:", duyuru_disi_ortalama_hiz_tum_yil)

    etki_orani_tum_yil = ((ortalama_hiz - duyuru_disi_ortalama_hiz_tum_yil) / duyuru_disi_ortalama_hiz_tum_yil) * 100
    print(f"\nTüm Yıl Bazında Etki Oranı: {etki_orani_tum_yil:.2f}%")

    print(uc_hafta_once_sonra_ortalama_hiz_dict[baslangic_tarihi])
    del uc_hafta_once_sonra_ortalama_hiz_dict[baslangic_tarihi]
    uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil = sum(uc_hafta_once_sonra_ortalama_hiz_dict.values()) / len(uc_hafta_once_sonra_ortalama_hiz_dict)

    del bes_hafta_once_sonra_ortalama_hiz_dict[baslangic_tarihi]
    bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil = sum(bes_hafta_once_sonra_ortalama_hiz_dict.values()) / len(bes_hafta_once_sonra_ortalama_hiz_dict)

    etki_orani_3_hafta_once_sonra = ((ortalama_hiz - uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) / uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) * 100
    print(f"3 Haftalık Etki Oranı: {etki_orani_3_hafta_once_sonra:.2f}%")

    etki_orani_5_hafta_once_sonra = ((ortalama_hiz - bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) / bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) * 100
    print(f"5 HaftalıkEtki Oranı: {etki_orani_5_hafta_once_sonra:.2f}%\n\n\n")

    etki_oranlari = {'DuyuruDisiOrtalamaHiz': duyuru_disi_ortalama_hiz_tum_yil,
                  'EtkiOraniTumYil': round(etki_orani_tum_yil, 2),
                  'EtkiOrani3HaftaOnceSonra': round(etki_orani_3_hafta_once_sonra, 2),
                  'EtkiOrani5HaftaOnceSonra': round(etki_orani_5_hafta_once_sonra, 2)}

    bolumler_ayrilmis_etki_orani_df.loc[i, etki_oranlari.keys()] = etki_oranlari.values()

In [ ]:
# Parçalanmış duyuruların önceki ve sonraki haftalarının hızlarına bakarak etki oranlarını df'e ekliyoruz.

for i in bolumler_ayrilmis_etki_orani_df.index:

    baslangic_tarihi = bolumler_ayrilmis_etki_orani_df.BaslangicTarihi[i]
    print("Duyuru Index:", bolumler_ayrilmis_etki_orani_df.DuyuruIndex[i])

    tarihListesi = []

    # Tarihten önceki haftaların o aynı günlerini bulmak için geriye doğru döngü
    tarih = baslangic_tarihi
    while (tarih - timedelta(days=7)).year == 2019:
        tarih -= timedelta(days=7)
        tarihListesi.append(tarih)

#     # Tarihten sonraki tarihleri bulmak için ileriye doğru döngü
#     tarih = baslangic_tarihi
#     while tarih.year == 2019:
#         tarihListesi.append(tarih)
#         tarih += timedelta(days=7)

#     print(tarihListesi)

    gunBas = bolumler_ayrilmis_etki_orani_df.BaslangicTarihi[i]
    saatBas = bolumler_ayrilmis_etki_orani_df.BaslangicSaati[i]

    gunBit = bolumler_ayrilmis_etki_orani_df.BitisTarihi[i]
    saatBit = bolumler_ayrilmis_etki_orani_df.BitisSaati[i]

    bolumler_ayrilmis_etki_orani_df['SegId'] = bolumler_ayrilmis_etki_orani_df['SegId'].round().astype(int)
    bolumler_ayrilmis_etki_orani_df['SegDir'] = bolumler_ayrilmis_etki_orani_df['SegDir'].round().astype(int)

    path = 'Veriler/Hız Verileri/2019/'
    file = str(bolumler_ayrilmis_etki_orani_df.SegId[i])
    dire = str(bolumler_ayrilmis_etki_orani_df.SegDir[i])
    a = path + file + '_' + dire + '_2019.csv'

    hizDos = pd.read_csv(a)
    hizDos.rename(columns = {'0':'date_time', '1':'speed'}, inplace = True)
    hizDos['date_time'] = pd.to_datetime(hizDos['date_time'], format='%Y-%m-%d %H:%M:%S')

    # Belirli saatlerin arasını df olarak al.
    tarihler_arasi_df = tarihArasiAlReferansTarih(gunBas, saatBas, gunBit, saatBit)
    #print(tarihler_arasi_df, "\n")

    satirSayisi = tarihler_arasi_df.shape[0]
    #print("Satir Sayisi:", satirSayisi)

    # "speed" sütunundaki değerlerin ortalamasını al.
    ortalama_hiz = tarihler_arasi_df['speed'].mean()

    ## Şimdi yılın diğer günlerine 7 gün arayla bakacağız.

    # Sözlük oluştur
    ortalama_hiz_dict = {}
    uc_hafta_once_sonra_ortalama_hiz_dict = {}
    bes_hafta_once_sonra_ortalama_hiz_dict = {}

    for j in tarihListesi:
        # Duyurunun yapıldığı tarih haftanın hangi gününe denk geliyorsa yıl boyunca o günlerin ilgili saat aralığını al.
        gunBas = j
        tarihler_arasi_df_yeni = tarihArasiAlDigerGunler(gunBas, saatBas, satirSayisi)

        satirSayisiYeni = tarihler_arasi_df_yeni.shape[0]

        ortalama_hiz_yeni = tarihler_arasi_df_yeni['speed'].mean()
        #print("Ortalama Hiz:", ortalama_hiz, "\n")

        # Sözlüğe ekle
        ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni

        # Etki oranını tüm yıl değil de k hafta almak için k haftanın değerlerini yeni sözlüklere de ekliyoruz.
        gun_farki = abs((gunBas - baslangic_tarihi).days)
        if gun_farki <= 21:
            uc_hafta_once_sonra_ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni
        if gun_farki <= 35:
            bes_hafta_once_sonra_ortalama_hiz_dict[gunBas] = ortalama_hiz_yeni

    # 3 sözlükteki boş verileri de siliyoruz.
    for key, value in list(ortalama_hiz_dict.items()):
        if value == -1.0:
            del ortalama_hiz_dict[key]

    for key, value in list(uc_hafta_once_sonra_ortalama_hiz_dict.items()):
        if value == -1.0:
            del uc_hafta_once_sonra_ortalama_hiz_dict[key]

    for key, value in list(bes_hafta_once_sonra_ortalama_hiz_dict.items()):
        if value == -1.0:
            del bes_hafta_once_sonra_ortalama_hiz_dict[key]

    sirali_sozluk = dict(sorted(ortalama_hiz_dict.items()))

    print("Duyuru Zamanindaki Ortalama Hiz:", ortalama_hiz, "\n")

    if (ortalama_hiz == -1.0):
        continue

    for key, value in list(ortalama_hiz_dict.items()):
        if key == baslangic_tarihi:
            del ortalama_hiz_dict[key]

    # Asıl tarihin artık bulunmadığı sözlükteki ortalama hızların ortalamasını al:
    duyuru_disi_ortalama_hiz_tum_yil = sum(ortalama_hiz_dict.values()) / len(ortalama_hiz_dict)
    print("Duyuru Zamani Disindaki Haftalarin, Haftanin O Gunune Ait Ortalama Hizi:", duyuru_disi_ortalama_hiz_tum_yil)

    etki_orani_tum_yil = ((ortalama_hiz - duyuru_disi_ortalama_hiz_tum_yil) / duyuru_disi_ortalama_hiz_tum_yil) * 100
    print(f"\nTüm Yıl Bazında Etki Oranı: {etki_orani_tum_yil:.2f}%")

    #print(uc_hafta_once_sonra_ortalama_hiz_dict[baslangic_tarihi])
    #del uc_hafta_once_sonra_ortalama_hiz_dict[baslangic_tarihi]
    uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil = sum(uc_hafta_once_sonra_ortalama_hiz_dict.values()) / len(uc_hafta_once_sonra_ortalama_hiz_dict)

    #del bes_hafta_once_sonra_ortalama_hiz_dict[baslangic_tarihi]
    bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil = sum(bes_hafta_once_sonra_ortalama_hiz_dict.values()) / len(bes_hafta_once_sonra_ortalama_hiz_dict)

    etki_orani_3_hafta_once_sonra = ((ortalama_hiz - uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) / uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) * 100
    print(f"3 Hafta Öncesine Göre Etki Oranı: {etki_orani_3_hafta_once_sonra:.2f}%")

    etki_orani_5_hafta_once_sonra = ((ortalama_hiz - bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) / bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) * 100
    print(f"5 Hafta Öncesine Göre Etki Oranı: {etki_orani_5_hafta_once_sonra:.2f}%\n\n\n")

    etki_oranlari = {'DuyuruDisiOrtalamaHiz': duyuru_disi_ortalama_hiz_tum_yil,
                  '5HaftaOnceyeKadarOrtalamaHiz': round(uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil, 2),
                  'EtkiOraniOncekilerTumYil': round(etki_orani_tum_yil, 2),
                  'EtkiOrani3Hafta': round(etki_orani_3_hafta_once_sonra, 2),
                  'EtkiOrani5HaftaOnce': round(etki_orani_5_hafta_once_sonra, 2)}

    bolumler_ayrilmis_etki_orani_df.loc[i, etki_oranlari.keys()] = etki_oranlari.values()

In [ ]:
# Başlangıç ve bitiş saati aynı olan duyuruları kaldırıyoruz çünkü her bir duyuru max birkaç saatlik olabilir yani aynı olamaz. (Bölümlere ayırdığımız için)
bolumler_ayrilmis_etki_orani_df = bolumler_ayrilmis_etki_orani_df[bolumler_ayrilmis_etki_orani_df['BaslangicSaati'] != bolumler_ayrilmis_etki_orani_df['BitisSaati']]

In [ ]:
# merged_bolunmus_df'e etki_orani_df'den Ilce ve YolAdresi bilgilerini ekliyoruz.
merged_bolunmus_df_uzunsuz = pd.merge(bolumler_ayrilmis_etki_orani_df, etki_orani_df[['XCoor', 'YCoor', 'Ilce', 'YolAdresi']], on=['XCoor', 'YCoor'], how='left')
merged_bolunmus_df_uzunsuz = merged_bolunmus_df_uzunsuz[merged_bolunmus_df_uzunsuz['DuyuruIndex'] != 28.0] # Bu duyurunun bitiş tarihi başlangıç tarihinden önce verildiği için kaldırdık.

In [ ]:
partOfDay = [[],[],[],[]]

for i in merged_bolunmus_df_uzunsuz.index:

    if(merged_bolunmus_df_uzunsuz.GununBolumu[i]=="Sabah"):
        partOfDay[0].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    if(merged_bolunmus_df_uzunsuz.GununBolumu[i]=="Ogle"):
        partOfDay[1].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    if(merged_bolunmus_df_uzunsuz.GununBolumu[i]=="Aksam"):
        partOfDay[2].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    if(merged_bolunmus_df_uzunsuz.GununBolumu[i]=="Gece"):
        partOfDay[3].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
partOfDayDict = dict()
for i in range(4):
    partOfDayDict[i] = sum(partOfDay[i]) / len(partOfDay[i])
    print(partOfDayDict[i])

parts = list(partOfDayDict.keys())
values = list(partOfDayDict.values())

colors = sns.color_palette("husl", len(parts))

sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=parts, y=values, palette=colors)

plt.xlabel('Günün Bölümleri')
plt.ylabel('Etki Oranı %')
plt.title('Yol Bakım Çalışmalarının Günün Bölümleri Bazında 5 Haftalık Ortalama Hızlara Etki Oranları (Duyurular Bölünmüş Şekilde Alındı)')

plt.xticks(parts, ['06:00-11:00 (Sabah)', '11:00-16:00 (Öğle)', '16:00-21:00 (Akşam)', '21:00-06:00 (Gece)'])

for i, v in enumerate(values):
    ax.text(i, v + 0.01, f"{len(partOfDay[i])} Satır\nMax: {np.max(partOfDay[i]):.2f}\nMin: {np.min(partOfDay[i]):.2f}", ha='center', va='bottom', color='white')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/GununBolumleri_5HaftalikEtkiOranlari-Duyurular Bölünmüş Şekilde ve Çok Uzun Duyuru Hariç Alınıyor")

# Aynı grafiğin İngilizce'sini çıkartacağız.

sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=parts, y=values, palette=colors)

plt.xlabel('Segments of the Day')
plt.ylabel('Impact Rate %')
plt.title('Impact Rates of Road Maintenance Works on 5-Week Average Speeds by Segments of the Day')

plt.xticks(parts, ['06:00-11:00 (Morning)', '11:00-16:00 (Noon)', '16:00-21:00 (Evening)', '21:00-06:00 (Night)'])

for i, v in enumerate(values):
    ax.text(i, v + 0.01, f"{len(partOfDay[i])} Row\nMax: {np.max(partOfDay[i]):.2f}\nMin: {np.min(partOfDay[i]):.2f}", ha='center', va='bottom', color='white')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/GununBolumleri_5HaftalikEtkiOranlari-Duyurular Bölünmüş Şekilde ve Çok Uzun Duyuru Hariç Alınıyor_EN")


In [ ]:
partOfDayforSpeed = [[],[],[],[]]

for i in merged_bolunmus_df_uzunsuz.index:
    if(merged_bolunmus_df_uzunsuz.GununBolumu[i]=="Sabah"):
        partOfDayforSpeed[0].append(merged_bolunmus_df_uzunsuz.DuyuruDisiOrtalamaHiz[i])
    elif(merged_bolunmus_df_uzunsuz.GununBolumu[i]=="Ogle"):
        partOfDayforSpeed[1].append(merged_bolunmus_df_uzunsuz.DuyuruDisiOrtalamaHiz[i])
    elif(merged_bolunmus_df_uzunsuz.GununBolumu[i]=="Aksam"):
        partOfDayforSpeed[2].append(merged_bolunmus_df_uzunsuz.DuyuruDisiOrtalamaHiz[i])
    elif(merged_bolunmus_df_uzunsuz.GununBolumu[i]=="Gece"):
        partOfDayforSpeed[3].append(merged_bolunmus_df_uzunsuz.DuyuruDisiOrtalamaHiz[i])

partOfDayforSpeedDict = dict()
for i in range(4):
    partOfDayforSpeedDict[i] = sum(partOfDayforSpeed[i]) / len(partOfDayforSpeed[i])
    print(partOfDayforSpeedDict[i])

parts = list(partOfDayforSpeedDict.keys())
values = list(partOfDayforSpeedDict.values())

colors = sns.color_palette("husl", len(parts))

sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=parts, y=values, palette=colors)

plt.xlabel('Günün Bölümleri')
plt.ylabel('Ortalama Hız')
plt.title('Günün Bölümleri Bazında Ortalama Hızlar (Duyurular Bölünmüş Şekilde Alındı)')

plt.xticks(parts, ['06:00-11:00 (Sabah)', '11:00-16:00 (Öğle)', '16:00-21:00 (Akşam)', '21:00-06:00 (Gece)'])

for i, v in enumerate(values):
    ax.text(i, v + 0.01, f"{len(partOfDayforSpeed[i])} Satır\nMax: {np.max(partOfDayforSpeed[i]):.2f}\nMin: {np.min(partOfDayforSpeed[i]):.2f}", ha='center', va='top', color='white')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/GununBolumleri_OrtalamaHizlar-Duyurular Bölünmüş Şekilde ve Çok Uzun Duyuru Hariç Alınıyor")

In [ ]:
# Duyuruları haftanın günleri bazında değerlendireceğiz.

weekdays = [[],[],[],[],[],[],[]]

for i in merged_bolunmus_df_uzunsuz.index:
    day = merged_bolunmus_df_uzunsuz.BaslangicTarihi[i].weekday()
    if(day==0):
         weekdays[0].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    elif(day==1):
         weekdays[1].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    elif(day==2):
         weekdays[2].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    elif(day==3):
         weekdays[3].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    elif(day==4):
         weekdays[4].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    elif(day==5):
         weekdays[5].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])
    elif(day==6):
         weekdays[6].append(merged_bolunmus_df_uzunsuz.EtkiOrani5HaftaOnceSonra[i])

weekdaysDict = dict()
for i in range(7):
    weekdaysDict[i] = sum(weekdays[i]) / len(weekdays[i])

# Sözlük verilerini sıralayarak x ve y değerlerini ayırıyoruz.
days = list(weekdaysDict.keys())
values = list(weekdaysDict.values())

# Renk paletini tanımlıyoruz.
colors = sns.color_palette("husl", len(days))

# Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluştuyoruz.
sns.set(style="whitegrid")
plt.figure(figsize=(10, 10))  # Grafik boyutunu genişletiyoruz.
ax = sns.barplot(x=days, y=values, palette=colors, saturation=0.8)  # Saturation parametresini ekliyoruz.

# Eksen ve başlık etiketlerini ekliyoruz.
plt.xlabel('Haftanın Günleri')
plt.ylabel('Etki Oranı %')
plt.title('Yol Bakım Çalışmalarının Haftanın Günleri Bazında 5 Haftalık Ortalama Hızlara Etki Oranları (Duyurular Bölünmüş Şekilde Alındı)')

# x ekseni etiketlerini güncelliyoruz.
plt.xticks(days, ['Pzt', 'Sal', 'Çar', 'Per', 'Cum', 'Cmt', 'Paz'])

# Sütunların üzerine değerleri yazdırıyoruz.
for i, v in enumerate(values):
    ax.text(i, v + 0.01, f"{len(weekdays[i])} Satır\nMax: {np.max(weekdays[i]):.2f}\nMin: {np.min(weekdays[i]):.2f}", ha='center', va='top', color='black')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/HaftaninGunleri_5HaftalikEtkiOranlari_GununBolumlerineAyrilmisHalde")

# Aynı grafiğin İngilizce'sini çıkartacağız.

# Seaborn ve Matplotlib'i kullanarak sütun grafiğini oluştuyoruz.
sns.set(style="whitegrid")
plt.figure(figsize=(10, 10))  # Grafik boyutunu genişletiyoruz.
ax = sns.barplot(x=days, y=values, palette=colors, saturation=0.8)  # Saturation parametresini ekliyoruz.

# Eksen ve başlık etiketlerini ekliyoruz.
plt.xlabel('Days of the Week')
plt.ylabel('Rate of Impact %')
plt.title('Impact Rates of Road Maintenance Works on 5-Week Average Speeds by Days of the Week')

# x ekseni etiketlerini güncelliyoruz.
plt.xticks(days, ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

# Sütunların üzerine değerleri yazdırıyoruz.
for i, v in enumerate(values):
    ax.text(i, v + 0.01, f"{len(weekdays[i])} Row\nMax: {np.max(weekdays[i]):.2f}\nMin: {np.min(weekdays[i]):.2f}", ha='center', va='top', color='black')

# Grafiği göster.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/HaftaninGunleri_5HaftalikEtkiOranlari_GununBolumlerineAyrilmisHalde_EN")

In [ ]:
from sklearn.preprocessing import LabelEncoder

# LabelEncoder oluşturuyoruz.
label_encoder = LabelEncoder()

# Bolum sütunundaki listeleri inceleyerek uygun sayısal değerleri atayacak fonksiyon:
def assign_numeric_value(bolum_list):
    if 'Sabah' in bolum_list:
        return 0
    elif 'Ogle' in bolum_list:
        return 1
    elif 'Aksam' in bolum_list:
        return 2
    elif 'Gece' in bolum_list:
        return 3
    else:
        return -1  # Eğer hiçbiri yoksa -1 döner.

merged_bolunmus_df_uzunsuz['GununBolumu_numeric'] = merged_bolunmus_df_uzunsuz['GununBolumu'].apply(assign_numeric_value)
merged_bolunmus_df_uzunsuz['YolAdresi_numeric'] = label_encoder.fit_transform(merged_bolunmus_df_uzunsuz['YolAdresi'])

merged_bolunmus_df_uzunsuz['BaslangicTarihi'] = pd.to_datetime(merged_bolunmus_df_uzunsuz['BaslangicTarihi'])
merged_bolunmus_df_uzunsuz['HaftaninGunu'] = merged_bolunmus_df_uzunsuz['BaslangicTarihi'].dt.day_name()

merged_bolunmus_df_uzunsuz = merged_bolunmus_df_uzunsuz.reset_index(drop=True)
merged_bolunmus_df_uzunsuz

In [ ]:
# Gün isimlerini Türkçe'ye çeviren fonksiyon:
def gunIsimleri(gun):
    if gun == 'Monday':
        gun = 'Pazartesi'
    elif gun == 'Tuesday':
        gun = 'Sali'
    elif gun == 'Wednesday':
        gun = 'Carsamba'
    elif gun == 'Thursday':
        gun = 'Persembe'
    elif gun == 'Friday':
        gun = 'Cuma'
    elif gun == 'Saturday':
        gun = 'Cumartesi'
    elif gun == 'Sunday':
        gun = 'Pazar'

    return gun

# 'HaftaninGunu' sütununu Türkçe'ye çeviriyoruz.
for i in merged_bolunmus_df_uzunsuz.index:
    merged_bolunmus_df_uzunsuz['HaftaninGunu'][i] = gunIsimleri(merged_bolunmus_df_uzunsuz['HaftaninGunu'][i])

merged_bolunmus_df_uzunsuz

In [ ]:
# Haftanın gününe göre değer döndüren fonksiyon:
def assign_numeric_value_day(day_list):
    if 'Pazartesi' in day_list:
        return 0
    elif 'Sali' in day_list:
        return 1
    elif 'Carsamba' in day_list:
        return 2
    elif 'Persembe' in day_list:
        return 3
    elif 'Cuma' in day_list:
        return 4
    elif 'Cumartesi' in day_list:
        return 5
    elif 'Pazar' in day_list:
        return 6
    else:
        return -1  # Eğer hiçbiri yoksa -1 döner.

merged_bolunmus_df_uzunsuz['HaftaninGunu_numeric'] = merged_bolunmus_df_uzunsuz['HaftaninGunu'].apply(assign_numeric_value_day)

In [ ]:
merged_bolunmus_df_uzunsuz

## Duyuruları Beşer Dakikalık Parçalara Ayırıyoruz:

In [ ]:
# Günün bölümleri için belirlenen saat sınırlarının duyuru içerisinde geçip geçmediğinin kontrolü:
def check_time(row):
    target_hours = [6, 11, 16, 21]
    return row.hour in target_hours and row.minute is 0

# Günün bölümünü bulma fonksiyonu:
def bul_bolum(baslangic_tarihi, baslangic_saat, bitis_tarihi, bitis_saat):
    baslangic_zaman = datetime.combine(baslangic_tarihi, baslangic_saat)
    bitis_zaman = datetime.combine(bitis_tarihi, bitis_saat)

    # Başlangıç ve bitiş arasındaki süreyi kontrol ediyoruz.
    fark = bitis_zaman - baslangic_zaman
    if fark.days >= 1:
        return list(bolumler.values())  # Eğer bir günden fazla süre varsa, tüm bölümleri al.
    else:
        bulunan_bolumler = set()
        for baslangic, bitis in bolumler.keys():
            if baslangic_zaman.time() < bitis:
                if baslangic < bitis_zaman.time() < bitis:
                    bulunan_bolumler.add(bolumler[(baslangic, bitis)])
                elif baslangic < (baslangic_zaman + timedelta(days=1)).time() < bitis:
                    bulunan_bolumler.add(bolumler[(baslangic, bitis)])
        return list(bulunan_bolumler) if bulunan_bolumler else None

def dfYeniSatirEkle(i, baslangicTarihi, baslangicSaati, bitisTarihi, bitisSaati, ortalamaHiz):
    yeni_satir = {'DuyuruIndex': mergedVhiz19.index[i],
                  'BaslangicTarihi': baslangicTarihi,
                  'BaslangicSaati': baslangicSaati,
                  'BitisTarihi': bitisTarihi,
                  'BitisSaati': bitisSaati,
                  'XCoor': mergedVhiz19.XCoor[i],
                  'YCoor': mergedVhiz19.YCoor[i],
                  'SegId': mergedVhiz19.vSegId[i],
                  'SegDir': mergedVhiz19.vSegDir[i],
                  'DuyuruZamaniOrtalamaHiz': ortalamaHiz,
                  'GununBolumu': bul_bolum(baslangicTarihi, baslangicSaati, bitisTarihi, bitisSaati)}

    return yeni_satir

# Dakikaları 5'e bölünecek biçimde almak için fonksiyon:
def dakikaYuvarla(gun_objesi, saat_objesi):
    # Saat verisini datetime objesine çevir
    saat_objesi = datetime.combine(gun_objesi, saat_objesi)

    # Dakikayı en yakın 5'e yuvarla
    if saat_objesi.minute % 5 != 0 or saat_objesi.second != 0:
        yuvarlanmis_dakika = (saat_objesi.minute // 5) * 5
    else:
        yuvarlanmis_dakika = saat_objesi.minute

    # Yeni dakika değerini datetime objesine ekle
    yeni_dakika = datetime(
        saat_objesi.year,
        saat_objesi.month,
        saat_objesi.day,
        saat_objesi.hour,
        yuvarlanmis_dakika)

    # Sonucu istediğiniz formata dönüştür
    sonuc = yeni_dakika.time()

    return sonuc

In [ ]:
# Duyuru verilerini beşer dakikalık aralıklarla bölen fonksiyon:
def split_announcement(row):
    start_date = datetime.combine(row['BaslangicTarihi'], row['BaslangicSaati'])
    end_date = datetime.combine(row['BitisTarihi'], row['BitisSaati'])

    # 5'er dakikalık aralıklarla tarih oluşturuyoruz.
    date_range = pd.date_range(start=start_date, end=end_date, freq='5T')
    #print(start_date, end_date)

    # Her bir tarih için yeni bir satır oluşturuyoruz.
    rows = []

    for date_time in date_range:
        new_row = row.copy()
        new_row['YeniBaslangicTarihi'] = date_time.date()
        new_row['YeniBaslangicSaati'] = dakikaYuvarla(date_time.date(), date_time.time())
        new_row['YeniBitisSaati'] = (datetime.combine(new_row['YeniBaslangicTarihi'], new_row['YeniBaslangicSaati']) + timedelta(minutes=5)).time()

        # Bitiş tarihi 00:00:00'a geçmişse, bir sonraki günü seç.
        if new_row['YeniBitisSaati'] == datetime.min.time():
            new_row['YeniBitisTarihi'] = date_time.date() + timedelta(days=1)
        else:
            new_row['YeniBitisTarihi'] = date_time.date()

        new_date = datetime.combine(new_row['YeniBaslangicTarihi'], new_row['YeniBaslangicSaati'])
        new_date_2 = new_date + timedelta(minutes=5)

        if(new_date >= start_date and new_date_2 <= end_date):
            rows.append(new_row)

    return pd.DataFrame(rows)

# Her bir duyuru için 5'er dakikalık aralıklarla yeni satırlar oluşturuyoruz.
beser_dklik_bolumlere_ayrilmis_df = pd.concat(merged_bolunmus_df_uzunsuz.apply(split_announcement, axis=1).tolist(), ignore_index=True)

# Gerekli olmayan sütunları çıkarıyoruz.
#beser_dklik_bolumlere_ayrilmis_df = beser_dklik_bolumlere_ayrilmis_df.drop(['DuyuruZamaniOrtalamaHiz', 'DuyuruDisiOrtalamaHiz', 'EtkiOraniTumYil', 'EtkiOrani3HaftaOnceSonra', 'EtkiOrani5HaftaOnceSonra', 'HaftaninGunu',  'HaftaninGunu_numeric'], axis=1)

In [ ]:
# Yeni sütun sırası:
new_column_order = ['DuyuruIndex', 'BaslangicTarihi', 'BaslangicSaati', 'BitisTarihi', 'BitisSaati',
                     'YeniBaslangicTarihi', 'YeniBaslangicSaati', 'YeniBitisTarihi', 'YeniBitisSaati',
                     'GununBolumu', 'Ilce', 'YolAdresi', 'SegId', 'SegDir', 'XCoor', 'YCoor',
                     'GununBolumu_numeric', 'YolAdresi_numeric']

# Sütunları yeniden düzenliyoruz.
beser_dklik_bolumlere_ayrilmis_df = beser_dklik_bolumlere_ayrilmis_df[new_column_order]

beser_dklik_bolumlere_ayrilmis_df

In [ ]:
# Duyurunun başlangıç saatini geçmiş ve gelecek haftaların aynı günü için alan fonksiyon:
def saatAlDigerGunler(baslangic_tarihi, baslangic_saati, hizDos):

    baslangic_tarihi_ve_saati = datetime.combine(baslangic_tarihi, baslangic_saati)

    # Tarih ve saat aralığındaki satırları seç
    secilen_satir = hizDos[(hizDos['date_time'] >= (baslangic_tarihi_ve_saati - timedelta(minutes=15))) &
                           (hizDos['date_time'] <= (baslangic_tarihi_ve_saati + timedelta(minutes=15)))]
    #print(secilen_satir)

    return secilen_satir

In [ ]:
# İki sütunun tipini Timestamp'tan datetime.date'e çeviriyoruz.
#beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicTarihi'] = beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicTarihi'].dt.date
beser_dklik_bolumlere_ayrilmis_df['BaslangicTarihi'] = beser_dklik_bolumlere_ayrilmis_df['BaslangicTarihi'].dt.date
# type(beser_dklik_bolumlere_ayrilmis_df.BaslangicTarihi[1])

In [ ]:
# Beşer dk'lık duyuruların hız karşılıklarını tabloya ekliyoruz.

b = 0
beser_dklik_bolumlere_ayrilmis_df['DateTime'] = None
beser_dklik_bolumlere_ayrilmis_df['Hiz'] = None

for i in beser_dklik_bolumlere_ayrilmis_df.index:

    print("Duyuru No: ", beser_dklik_bolumlere_ayrilmis_df.DuyuruIndex[i])
    print("Başlangıç Tarihi: ", beser_dklik_bolumlere_ayrilmis_df.YeniBaslangicTarihi[i])

    tarihListesi = []
    diger_hizlar = {}
    uc_hafta_once_sonra_hiz_dict = {}
    bes_hafta_once_sonra_hiz_dict = {}

    # Duyuruya ait hız verilerinin olduğu dosyayı buluyoruz.
    path = 'Veriler/Hız Verileri/2019/'
    file = str(beser_dklik_bolumlere_ayrilmis_df.SegId[i])
    dire = str(beser_dklik_bolumlere_ayrilmis_df.SegDir[i])
    a = path + file + '_' + dire + '_2019.csv'

    if(a!=b):
        hizDos = pd.read_csv(a)
        hizDos.rename(columns = {'0':'date_time', '1':'speed'}, inplace = True)
        hizDos['date_time'] = pd.to_datetime(hizDos['date_time'], format='%Y-%m-%d %H:%M:%S')

    beser_dklik_bolumlere_ayrilmis_df['DateTime'][i] = datetime.combine(beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicTarihi'][i], beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicSaati'][i])

    # Her bir satır için uygun speed değerini df'e ekliyoruz.
    current_datetime = beser_dklik_bolumlere_ayrilmis_df['DateTime'][i]
    matching_speed = float(hizDos[hizDos['date_time'] == current_datetime]['speed'])
    beser_dklik_bolumlere_ayrilmis_df['Hiz'][i] = round(matching_speed, 2)

    if(beser_dklik_bolumlere_ayrilmis_df['Hiz'][i] == -1.0):
        continue

    b = a

    gunBas = beser_dklik_bolumlere_ayrilmis_df.YeniBaslangicTarihi[i]
    saatBas = beser_dklik_bolumlere_ayrilmis_df.YeniBaslangicSaati[i]
    gunSaat = beser_dklik_bolumlere_ayrilmis_df.DateTime[i]

    # Tarihten önceki haftaların aynı günlerini bulmak için geriye doğru döngü:
    tarih = gunBas
    while (tarih - timedelta(days=7)).year == 2019:
        tarih -= timedelta(days=7)
        tarihListesi.append(tarih)

    #print(tarihListesi)

    for j in tarihListesi:
        # Duyurunun yapıldığı tarih haftanın hangi gününe denk geliyorsa yıl boyunca o günlerin ilgili saatini alıyoruz.
        gun = j
        # Saat ve dakikayı al
        saat = saatBas.hour
        dakika = saatBas.minute

        secilen_satir = saatAlDigerGunler(gun, saatBas, hizDos)
        #print(secilen_satir)

        # Sözlüğe ekle.
        diger_hizlar[gun] = secilen_satir.iloc[3]['speed']

        # Etki oranını tüm yıl değil de k hafta almak için k haftanın değerlerini yeni sözlüklere de ekliyoruz.
        gun_farki = abs((gun - gunBas).days)
        if gun_farki <= 21:
            uc_hafta_once_sonra_hiz_dict[gun] = secilen_satir['speed'].mean()
        if gun_farki <= 35:
            bes_hafta_once_sonra_hiz_dict[gun] = secilen_satir['speed'].mean()

    diger_hizlar = {key: value for key, value in diger_hizlar.items() if value == value}
    uc_hafta_once_sonra_hiz_dict = {key: value for key, value in uc_hafta_once_sonra_hiz_dict.items() if value == value}
    bes_hafta_once_sonra_hiz_dict = {key: value for key, value in bes_hafta_once_sonra_hiz_dict.items() if value == value}

    sirali_sozluk = dict(sorted(diger_hizlar.items()))

    print("Duyuru Zamanindaki Hiz:", round(float(matching_speed), 2))

    for key, value in list(diger_hizlar.items()):
        if key == gunBas:
            del diger_hizlar[key]

    # Asıl tarihin artık bulunmadığı sözlükteki hızların ortalamasını al:
    duyuru_disi_ortalama_hiz_tum_yil = sum(diger_hizlar.values()) / len(diger_hizlar)
    print("Duyuru Zamani Disindaki Haftalarin, Haftanin O Gunune Ait Ortalama Hizi:", round(duyuru_disi_ortalama_hiz_tum_yil, 2))

    uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil = sum(uc_hafta_once_sonra_hiz_dict.values()) / len(uc_hafta_once_sonra_hiz_dict)

    bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil = sum(bes_hafta_once_sonra_hiz_dict.values()) / len(bes_hafta_once_sonra_hiz_dict)

    etki_orani_tum_yil = ((matching_speed - duyuru_disi_ortalama_hiz_tum_yil) / duyuru_disi_ortalama_hiz_tum_yil) * 100
    print("Tüm Yıl Bazında Etki Oranı: ", round(float(etki_orani_tum_yil), 2), "%")

    etki_orani_3_hafta_once_sonra = ((matching_speed - uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) / uc_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) * 100
    print("3 Haftalık Etki Oranı:", round(float(etki_orani_3_hafta_once_sonra), 2), "%")

    etki_orani_5_hafta_once_sonra = ((matching_speed - bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) / bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil) * 100
    print("5 Haftalık Etki Oranı:", round(float(etki_orani_5_hafta_once_sonra), 2), "%\n")

    satirlar = {'5HaftaOncesineKadarOrtalamaHiz': round(bes_hafta_once_sonra_duyuru_disi_ortalama_hiz_tum_yil, 2),
                'EtkiOrani5HaftaOncesineGore': round(etki_orani_5_hafta_once_sonra, 2)}

    beser_dklik_bolumlere_ayrilmis_df.loc[i, satirlar.keys()] = satirlar.values()


In [ ]:
beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicTarihi'] = pd.to_datetime(beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicTarihi'])
beser_dklik_bolumlere_ayrilmis_df['HaftaninGunu'] = beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicTarihi'].dt.day_name()

# 'HaftaninGunu' sütununu Türkçe'ye çeviriyoruz.
for i in beser_dklik_bolumlere_ayrilmis_df.index:
    beser_dklik_bolumlere_ayrilmis_df['HaftaninGunu'][i] = gunIsimleri(beser_dklik_bolumlere_ayrilmis_df['HaftaninGunu'][i])

merged_bolunmus_df_uzunsuz

beser_dklik_bolumlere_ayrilmis_df['HaftaninGunu_numeric'] = beser_dklik_bolumlere_ayrilmis_df['HaftaninGunu'].apply(assign_numeric_value_day)

In [ ]:
beser_dklik_bolumlere_ayrilmis_df = beser_dklik_bolumlere_ayrilmis_df[beser_dklik_bolumlere_ayrilmis_df['Hiz'] != -1.0]

## Harita Gösterimleri

In [ ]:
ilceler = etki_orani_df['Ilce'].unique()  # Veri içerisindeki unique ilçe isimlerini alıyoruz.
ilce_dict = {}

X_dict = {}
Y_dict = {}

for ilce in ilceler:
    ilce_satirlari = etki_orani_df[etki_orani_df['Ilce'] == ilce]
    etki_oranlari = ilce_satirlari['EtkiOrani5HaftaOnceSonra'].tolist()
    ilce_dict[ilce] = etki_oranlari

ilce_ortalama = {ilce: sum(etki_oranlari) / len(etki_oranlari) for ilce, etki_oranlari in ilce_dict.items()}

for ilce in ilceler:
    ilce_satirlari = etki_orani_df[etki_orani_df['Ilce'] == ilce]
    X_coords = ilce_satirlari['XCoor'].tolist()
    X_dict[ilce] = X_coords

X_ortalama = {X: sum(X_coords) / len(X_coords) for X, X_coords in X_dict.items()}

for ilce in ilceler:
    ilce_satirlari = etki_orani_df[etki_orani_df['Ilce'] == ilce]
    Y_coords = ilce_satirlari['YCoor'].tolist()
    Y_dict[ilce] = Y_coords

Y_ortalama = {Y: sum(Y_coords) / len(Y_coords) for Y, Y_coords in Y_dict.items()}

ilce_bazinda_df = pd.DataFrame(list(X_ortalama.items()), columns=['Ilce', 'XCoor'])

df_Y_coords = pd.DataFrame(list(Y_ortalama.items()), columns=['Ilce', 'YCoor'])

ilce_bazinda_df = pd.merge(ilce_bazinda_df, df_Y_coords, on='Ilce')

df_ilce_etki_orani = pd.DataFrame(list(ilce_ortalama.items()), columns=['Ilce', 'EtkiOrani'])

ilce_bazinda_df = pd.merge(ilce_bazinda_df, df_ilce_etki_orani, on='Ilce')

In [ ]:
ilce_bazinda_df

In [ ]:
# Harita gösterimi için merkezi belirliyoruz.
X = mergedVhiz19[(mergedVhiz19['vSegId'] == 440) & (mergedVhiz19['vSegDir'] == 1)].XCoor.mean()
Y = mergedVhiz19[(mergedVhiz19['vSegId'] == 440) & (mergedVhiz19['vSegDir'] == 1)].YCoor.mean()

center = X, Y

In [ ]:
## İlçe Bazında Etki Oranlarının Haritada Gösterimi

if not (pd.isna(X) or pd.isna(Y)):
    center = X, Y

    m = Map(center=center, zoom=10)

    # Haritayı oluşturuyoruz.
    for i in ilce_bazinda_df.index:
        ilce = ilce_bazinda_df.Ilce[i]
        corX = ilce_bazinda_df.XCoor[i]
        corY = ilce_bazinda_df.YCoor[i]
        etki_orani = ilce_bazinda_df.EtkiOrani[i]

        # Marker oluşturuyoruz ve etki oranıyla yol adresini içeren HTML widget'ını ekliyoruz.
        marker = Marker(location=(corX, corY), draggable=True)
        html_widget = HTML(value=f'District: {ilce}<br>Impact Rate: {etki_orani:.2f}')
        marker.popup = html_widget

        # Haritaya ekliyoruz.
        m.add_layer(marker)
        circle = Circle()
        circle.radius = 1
        circle.color = "red"
        circle.fill_color = "red"
        circle.location = corX, corY
        m.add_layer(circle)

    display(m)

In [ ]:
# İlçe Bazında Etki Oranlarının Renklerle Birlikte Haritada Gösterimi

if not (pd.isna(X) or pd.isna(Y)):
    center = X, Y

    m = Map(center=center, zoom=10)

    # Renk kategorilerini belirliyoruz.
    green_threshold = 0
    yellow_threshold = -20

    for i in ilce_bazinda_df.index:
        ilce = ilce_bazinda_df.Ilce[i]
        corX = ilce_bazinda_df.XCoor[i]
        corY = ilce_bazinda_df.YCoor[i]
        etki_orani = ilce_bazinda_df.EtkiOrani[i]

        marker = Marker(location=(corX, corY), draggable=True)
        html_widget = HTML(value=f'District: {ilce}<br>Impact Rate: {etki_orani:.2f}')

        marker = Marker(location=(corX, corY), draggable=True)

        # Etki oranına göre renk belirliyoruz.
        if etki_orani >= green_threshold:
            marker.icon = AwesomeIcon(name='info', marker_color='green')
        elif etki_orani >= yellow_threshold:
            marker.icon = AwesomeIcon(name='info', marker_color='blue')
        else:
            marker.icon = AwesomeIcon(name='info', marker_color='red')

        marker.popup = html_widget
        m.add_layer(marker)

        # Circle ekliyoruz.
        circle = Circle()
        circle.radius = 1
        circle.color = "red"
        circle.fill_color = "red"
        circle.location = corX, corY
        m.add_layer(circle)

    # Haritayı göster.
    display(m)

In [ ]:
yollar = etki_orani_df['YolAdresi'].unique()  # Veri içerisindeki unique yol isimlerini alıyoruz.

X_dict = {}
Y_dict = {}

for yol in yollar:
    yol_satirlari = etki_orani_df[etki_orani_df['YolAdresi'] == yol]
    X_coords = yol_satirlari['XCoor'].tolist()
    X_dict[yol] = X_coords

X_ortalama = {X: sum(X_coords) / len(X_coords) for X, X_coords in X_dict.items()}

for yol in yollar:
    yol_satirlari = etki_orani_df[etki_orani_df['YolAdresi'] == yol]
    Y_coords = yol_satirlari['YCoor'].tolist()
    Y_dict[yol] = Y_coords

Y_ortalama = {Y: sum(Y_coords) / len(Y_coords) for Y, Y_coords in Y_dict.items()}

yol_bazinda_df = pd.DataFrame(list(X_ortalama.items()), columns=['Yol', 'XCoor'])

df_Y_coords = pd.DataFrame(list(Y_ortalama.items()), columns=['Yol', 'YCoor'])

yol_bazinda_df = pd.merge(yol_bazinda_df, df_Y_coords, on='Yol')

df_yol_etki_orani = pd.DataFrame(list(yol_adresi_ortalama.items()), columns=['Yol', 'EtkiOrani'])

yol_bazinda_df = pd.merge(yol_bazinda_df, df_yol_etki_orani, on='Yol')

In [ ]:
yol_bazinda_df

In [ ]:
# Yol ve Yön Bazında Etki Oranlarının Haritada Gösterimi (Koordinatların Ortalaması ile)

# Harita merkezi ve başlangıç zoom düzeyini belirliyoruz.
center = (41, 29)
zoom_level = 10

# Map oluşturuyoruz.
m = Map(center=center, zoom=zoom_level)

# Renk kategorilerini belirliyoruz.
green_threshold = 0
yellow_threshold = -20

for i in yol_bazinda_df.index:
    yol = yol_bazinda_df.Yol[i]
    corX = yol_bazinda_df.XCoor[i]
    corY = yol_bazinda_df.YCoor[i]
    etki_orani = yol_bazinda_df.EtkiOrani[i]
    direction = yol_bazinda_df.Yol[i].split(":")[1]

    marker = Marker(location=(corX, corY), draggable=True)
    html_widget = HTML(value=f'Yol: {yol}<br>Etki Oranı: {etki_orani:.2f}')

    if direction == '0':
        if etki_orani >= green_threshold:
            marker.icon = AwesomeIcon(name='arrow-left', marker_color='green')
        elif etki_orani >= yellow_threshold:
            marker.icon = AwesomeIcon(name='arrow-left', marker_color='blue')
        else:
            marker.icon = AwesomeIcon(name='arrow-left', marker_color='red')
    elif direction == '1':
        if etki_orani >= green_threshold:
            marker.icon = AwesomeIcon(name='arrow-right', marker_color='green')
        elif etki_orani >= yellow_threshold:
            marker.icon = AwesomeIcon(name='arrow-right', marker_color='blue')
        else:
            marker.icon = AwesomeIcon(name='arrow-right', marker_color='red')

    marker.popup = html_widget
    m.add_layer(marker)

# Haritayı göster.
display(m)

In [ ]:
# Yol ve Yön Bazında Etki Oranlarının Haritada Gösterimi (Tüm Duyurular Tek Tek)

if not (pd.isna(X) or pd.isna(Y)):
    center = X, Y

    m = Map(center=center, zoom=10)

    for i in etki_orani_df.index:
        corX = etki_orani_df.XCoor[i]
        corY = etki_orani_df.YCoor[i]
        etki_orani = etki_orani_df.EtkiOraniTumYil[i]
        yol_adresi = etki_orani_df.YolAdresi[i]  # Yol adresini alın
        direction = etki_orani_df.YolAdresi[i].split(":")[1]

        marker = Marker(location=(corX, corY), draggable=True)
        html_widget = HTML(value=f'Roadway: {yol_adresi}<br>Impact Rate: {etki_orani:.2f}')

        if direction == '0':
            if etki_orani >= green_threshold:
                marker.icon = AwesomeIcon(name='arrow-left', marker_color='green')
            elif etki_orani >= yellow_threshold:
                marker.icon = AwesomeIcon(name='arrow-left', marker_color='blue')
            else:
                marker.icon = AwesomeIcon(name='arrow-left', marker_color='red')
        elif direction == '1':
            if etki_orani >= green_threshold:
                marker.icon = AwesomeIcon(name='arrow-right', marker_color='green')
            elif etki_orani >= yellow_threshold:
                marker.icon = AwesomeIcon(name='arrow-right', marker_color='blue')
            else:
                marker.icon = AwesomeIcon(name='arrow-right', marker_color='red')

        marker.popup = html_widget
        m.add_layer(marker)

    display(m)
else:
    print("Cannot create map. X or Y is NaN.")

## Regresyon

In [ ]:
from sklearn.model_selection import train_test_split # for splitting the data
from sklearn.metrics import mean_squared_error # for calculating the cost function
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

In [ ]:
# Train ve test df'lerini rastgele ayırıyoruz.

x = beser_dklik_bolumlere_ayrilmis_df[['5HaftaOncesineKadarOrtalamaHiz', 'GununBolumu_numeric', 'HaftaninGunu_numeric', 'XCoor', 'YCoor']]
y = beser_dklik_bolumlere_ayrilmis_df['Hiz']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Modeli oluşturuyoruz.
model = RandomForestRegressor(random_state=25)

# Modeli eğitiyoruz.
model.fit(x_train, y_train)

# Test seti üzerinde tahmin yaptırıyoruz.
y_pred = model.predict(x_test)

def mape(y_true, y_pred):
    errors = abs((y_true - y_pred) / y_true)
    mape_value = 100 * np.mean(errors)
    return mape_value

mape_value = mape(y_test, y_pred)
print("MAPE: {:.2f}%".format(mape_value))

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('Gerçek Değerler')
plt.ylabel('Tahmin Edilen Değerler')
plt.title('Gerçek vs Tahmin Edilen Hız Değerleri - RFR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/RFRegressor_y_Hiz_train_test_split")

In [ ]:
# Train test veri setlerini duyurunu indexinden ayırıyoruz.
# %56,5 train, %43,5 test

# Train verilerini seçiyoruz.
train_df = beser_dklik_bolumlere_ayrilmis_df.iloc[:1585]
# Test verilerini seçiyoruz.
test_df = beser_dklik_bolumlere_ayrilmis_df.iloc[1585:]
test_df = test_df.reset_index()

# Feature Selection
featureCols = ['5HaftaOncesineKadarOrtalamaHiz', 'GununBolumu_numeric', 'HaftaninGunu_numeric', 'XCoor', 'YCoor']
x_train = train_df[featureCols]
y_train = train_df['Hiz']

x_test = test_df[featureCols]
y_test = test_df['Hiz']

# Modeli oluşturuyoruz.
model = RandomForestRegressor(n_estimators=250, random_state = 15)

# Modeli eğitiyoruz.
model.fit(x_train, y_train)

# Test seti üzerinde tahmin yaptırıyoruz.
y_pred = model.predict(x_test)

def mape(y_true, y_pred):
    errors = abs((y_true - y_pred) / y_true)
    mape_value = 100 * np.mean(errors)
    return mape_value

mape_value = mape(y_test, y_pred)
print("MAPE: {:.2f}%\n".format(mape_value))

# Boş bir liste oluşturuyoruz.
mape_values = []

# Test seti üzerinde tahmin yaptırıyoruz.
y_preds = model.predict(x_test)

tekli_duyuru_mape_df = pd.DataFrame()

# Her bir test örneği için ayrı MAPE hesaplıyoruz.
for i in range(len(y_test)):
    y_true_i = y_test.iloc[i]
    y_pred_i = y_preds[i]
    mape_i = mape(y_true_i, y_pred_i)
    mape_values.append(mape_i)
    #print("Test Örneği {} - Duyuru {} - MAPE: {:.2f}%".format(i+1, test_df.DuyuruIndex[i], mape_i))

    veriler = {'DuyuruIndex': test_df.DuyuruIndex[i],
               'YeniBaslangicTarihi': test_df.YeniBaslangicTarihi[i],
               'YeniBaslangicSaati': test_df.YeniBaslangicSaati[i],
               'GercekHiz': y_true_i,
               'TahminEdilenHiz': y_pred_i,
               'MAPEDegeri': mape_i}

    tekli_duyuru_mape_df.loc[i, veriler.keys()] = veriler.values()


print("Max MAPE: {:.2f}%".format(max(mape_values)))
print("Min MAPE: {:.2f}%\n".format(min(mape_values)))

# Duyuru numaralarını unique olarak alıyoruz.
unique_duyuru_nums = test_df['DuyuruIndex'].unique()

# Ortalama MAPE değerini hesaplıyoruz.
average_mape = np.mean(mape_values)

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('Gerçek Değerler')
plt.ylabel('Tahmin Edilen Değerler')
plt.title('Gerçek vs Tahmin Edilen Hız Değerleri - RFR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/Genel_RFRegressor_x_{}_y_Hiz".format(featureCols))

# İngilizce Grafik

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Actual and Predicted Speed Values - RFR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/Genel_RFRegressor_x_{}_y_Hiz_EN".format(featureCols))

In [ ]:
beser_dklik_bolumlere_ayrilmis_df = beser_dklik_bolumlere_ayrilmis_df.drop(['DateTime'], axis=1)

In [ ]:
tekli_duyuru_mape_df.YeniBaslangicTarihi = pd.to_datetime(tekli_duyuru_mape_df.YeniBaslangicTarihi)
tekli_duyuru_mape_df.YeniBaslangicTarihi = tekli_duyuru_mape_df.YeniBaslangicTarihi.dt.date

In [ ]:
beser_dklik_bolumlere_ayrilmis_df.YeniBaslangicTarihi = pd.to_datetime(beser_dklik_bolumlere_ayrilmis_df.YeniBaslangicTarihi)
beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicTarihi'] = beser_dklik_bolumlere_ayrilmis_df['YeniBaslangicTarihi'].dt.date
beser_dklik_bolumlere_ayrilmis_df = pd.merge(beser_dklik_bolumlere_ayrilmis_df, tekli_duyuru_mape_df[['DuyuruIndex', 'YeniBaslangicTarihi', 'YeniBaslangicSaati', 'GercekHiz', 'TahminEdilenHiz', 'MAPEDegeri']], on=['DuyuruIndex', 'YeniBaslangicTarihi', 'YeniBaslangicSaati'], how='left')

In [ ]:
beser_dklik_bolumlere_ayrilmis_df

In [ ]:
df_ortalamalar = tekli_duyuru_mape_df.groupby('DuyuruIndex').mean().reset_index()
df_ortalamalar

In [ ]:
# Test veri setindeki her bir duyuru index için ayrı bir grafik oluşturacağız.
# Bu grafikte duyurunun beşer dk'lık her parçasının tahminindeki hata oranını göreceğiz.

# MAPE değeri boş olmayan satırları seçiyoruz.
df_non_null_mape = beser_dklik_bolumlere_ayrilmis_df.dropna(subset=['MAPEDegeri'])

# Her bir duyuru index için grafik oluşturacağız.
unique_duyuru_index = df_non_null_mape['DuyuruIndex'].unique()

for duyuru_index in unique_duyuru_index:

    # Duyuru indexe sahip satırları seçiyoruz.
    df_duyuru = df_non_null_mape[df_non_null_mape['DuyuruIndex'] == duyuru_index]
    print(type(df_duyuru['YeniBaslangicSaati']))

    parcalar = list(df_duyuru.index)
    degerler = list(df_duyuru['MAPEDegeri'])
    ortMAPE = df_duyuru['MAPEDegeri'].mean()

    # Bir figür ve alt eksen oluştuyoruz.
    fig, ax = plt.subplots(figsize=(10, 6))

    # Çizgi grafiği ekliyoruz.
    ax.plot(parcalar, degerler, marker='o', linestyle='-')

    # Grafik özellikleri:
    ax.set_title('{} No\'lu Duyuru Parçası İçin MAPE Değerleri - Ortalama MAPE: {:.2f}%'.format(duyuru_index, ortMAPE))
    ax.set_xlabel('Duyuru Parçası')
    ax.set_ylabel('MAPE Değeri %')
    ax.grid(True)

    # Grafiği göster.
    plt.show()

    ax_fig = ax.get_figure()
    ax_fig.savefig("Grafikler/MAPE/{} No'lu Duyuru için MAPE Değerleri.png".format(duyuru_index))
    
    # İngilizce Grafik
    
    # Bir figür ve alt eksen oluştuyoruz.
    fig, ax = plt.subplots(figsize=(10, 6))

    # Çizgi grafiği ekliyoruz.
    ax.plot(parcalar, degerler, marker='o', linestyle='-')

    # Grafik özellikleri:
    ax.set_title('MAPE Values for Announcement Part No. {} - General MAPE: {:.2f}%'.format(duyuru_index, ortMAPE))
    ax.set_xlabel('Announcement Part')
    ax.set_ylabel('MAPE Values %')
    ax.grid(True)

    # Grafiği göster.
    plt.show()

    ax_fig = ax.get_figure()
    ax_fig.savefig("Grafikler/MAPE/{} No'lu Duyuru için MAPE Değerleri_EN.png".format(duyuru_index))  

In [ ]:
# Tüm duyuruların ortalama MAPE değerlerini sütun grafiğinde göstereceğiz.

colors = sns.color_palette("husl", len(parts))

# Sütun grafiği oluşturuyoruz.
plt.figure(figsize=(15, 8))
ax = sns.barplot(x='DuyuruIndex', y='MAPEDegeri', data=df_ortalamalar, ci=None, palette = colors)
plt.xlabel('Duyuru Index')
plt.ylabel('Ortalama MAPE')
plt.title('Her Duyuru için Ortalama MAPE Değerleri')

plt.xticks(rotation=90)
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/Ortalama_MAPEler")

# İngilizce Grafik

# Sütun grafiği oluşturuyoruz.
plt.figure(figsize=(15, 8))
ax = sns.barplot(x='DuyuruIndex', y='MAPEDegeri', data=df_ortalamalar, ci=None, palette = colors)
plt.xlabel('Announcement Index')
plt.ylabel('MAPE')
plt.title('MAPE Values for Announcements')

plt.xticks(rotation=90)
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/Ortalama_MAPEler_EN")

In [ ]:
filtre = ~beser_dklik_bolumlere_ayrilmis_df['DuyuruIndex'].isin([58.0, 66.0, 74.0, 76.0])
filtrelenmis_df = beser_dklik_bolumlere_ayrilmis_df[filtre]

In [ ]:
filtrelenmis_df = filtrelenmis_df.reset_index()
filtrelenmis_df

In [ ]:
# Train test veri setlerini duyurunu indexinden ayırıyoruz.
# %56,5 train, %43,5 test

# Train verilerini seçiyoruz.
train_df = filtrelenmis_df.iloc[:1426]
# Test verilerini seçiyoruz.
test_df = filtrelenmis_df.iloc[1426:]
test_df = test_df.reset_index()

# Feature Selection
featureCols = ['5HaftaOncesineKadarOrtalamaHiz', 'GununBolumu_numeric', 'HaftaninGunu_numeric', 'XCoor', 'YCoor']
x_train = train_df[featureCols]
y_train = train_df['Hiz']

x_test = test_df[featureCols]
y_test = test_df['Hiz']

# Modeli oluşturuyoruz.
model = RandomForestRegressor(n_estimators=250, random_state = 15)

# Modeli eğitiyoruz.
model.fit(x_train, y_train)

# Test seti üzerinde tahmin yaptırıyoruz.
y_pred = model.predict(x_test)

def mape(y_true, y_pred):
    errors = abs((y_true - y_pred) / y_true)
    mape_value = 100 * np.mean(errors)
    return mape_value

mape_value = mape(y_test, y_pred)
print("MAPE: {:.2f}%\n".format(mape_value))

# Test seti üzerinde tahmin yaptırıyoruz.
y_preds = model.predict(x_test)

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('Gerçek Değerler')
plt.ylabel('Tahmin Edilen Değerler')
plt.title('Gerçek vs Tahmin Edilen Hız Değerleri - RFR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/Genel_RFRegressor_x_{}_y_Hiz_Outlierlarsız".format(featureCols))

# İngilizce Grafik

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Actual and Predicted Speed Values - RFR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/Genel_RFRegressor_x_{}_y_Hiz_Outlierlarsız_EN".format(featureCols))

In [ ]:
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

def increase_widget_sizes(widget, font_size=12, width=20):
    style = ttk.Style()
    style.configure("TLabel", font=('Helvetica', font_size), foreground='#333333')
    style.configure("TButton", font=('Helvetica', font_size), width=width)
    style.configure("TEntry", font=('Helvetica', font_size), width=width)
    style.configure("Treeview.Heading", font=('Helvetica', font_size), foreground='#333333')
    style.configure("Treeview", font=('Helvetica', font_size))

def filter_data():
    filter_value = filter_entry.get()
    if filter_value:
        filtered_df = test_df[test_df['DuyuruIndex'] == int(filter_value)]
        update_treeview(filtered_df)
        plot_graph(filtered_df)
    else:
        update_treeview(test_df)

def update_treeview(data):
    test_data_treeview.delete(*test_data_treeview.get_children())
    for i, row in data.iterrows():
        test_data_treeview.insert('', i, values=(i, row['DuyuruIndex'], row['YeniBaslangicTarihi'], row['YeniBaslangicSaati'], row['5HaftaOncesineKadarOrtalamaHiz'], row['Ilce'], row['YolAdresi']))

def on_item_select(event):
    selected_item = test_data_treeview.selection()[0]
    selected_index = int(float(test_data_treeview.item(selected_item, 'values')[0]))

    # Seçilen veriyi güncelliyoruz.
    selected_data_label.config(text=f"Seçilen Veri: {selected_index}")

    # Seçilen veriyi kullanarak gerçek hız, tahmin edilen hız ve MAPE değerlerini alıyoruz.
    gercek_hiz = test_df.at[selected_index, 'GercekHiz']
    tahmin_edilen_hiz = test_df.at[selected_index, 'TahminEdilenHiz']
    mape_value = test_df.at[selected_index, 'MAPEDegeri']

    # Tahmin sonuçlarını ve hata oranını güncelliyoruz.
    result_label.config(text=f"Gerçek Hız Değeri: {gercek_hiz:.2f}\n\nTahmin Edilen Hız Değeri: {tahmin_edilen_hiz:.2f}\n\nMAPE Değeri: {mape_value:.2f}%")

def plot_graph(df_duyuru):
    # Mevcut grafiği temizliyoruz.
    for widget in graph_frame.winfo_children():
        widget.destroy()

    # Grafiği oluşturuyoruz.
    fig, ax = plt.subplots(figsize=(10, 6))
    parcalar = list(df_duyuru.index)
    degerler = list(df_duyuru['MAPEDegeri'])
    ax.plot(parcalar, degerler, marker='o', linestyle='-')
    ax.set_title('Seçilen Duyuru İçin MAPE Değerleri')
    ax.set_xlabel('Duyuru Parçası')
    ax.set_ylabel('MAPE Değeri %')
    ax.grid(True)

    # Matplotlib figürünü Tkinter canvasına yerleştiriyoruz.
    canvas = FigureCanvasTkAgg(fig, master=graph_frame)
    canvas.draw()
    canvas.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=1)

# Tkinter uygulamasını oluşturuyoruz.
app = tk.Tk()
app.title("Hız Tahmini Arayüzü")

# Pencere boyutunu belirliyoruz.
app.geometry("1000x600")

# Arka plan rengini belirliyoruz.
app.configure(bg='#f0f0f0')

# Test verilerini gösteren bir LabelFrame oluşturuyoruz.
test_data_frame = ttk.LabelFrame(app, text="Test Verileri", padding=(10, 10))
test_data_frame.grid(row=0, column=0, padx=10, pady=10, sticky="wens")

# Test verilerini içeren bir Treeview oluşturuyoruz.
test_data_treeview = ttk.Treeview(test_data_frame, columns=('DfIndex', 'DuyuruIndex', 'YeniBaslangicTarihi', 'YeniBaslangicSaati', '5HaftaOncesineKadarOrtalamaHiz', 'Ilce', 'YolAdresi'))
test_data_treeview.heading('#1', text='Df Index')
test_data_treeview.heading('#2', text='Duyuru Index')
test_data_treeview.heading('#3', text='Başlangıç Tarihi')
test_data_treeview.heading('#4', text='Başlangış Saati')
test_data_treeview.heading('#5', text='5 Hafta Öncesine Kadar Ortalama Hiz')
test_data_treeview.heading('#6', text='İlçe')
test_data_treeview.heading('#7', text='Yol Adresi')

test_df = beser_dklik_bolumlere_ayrilmis_df.iloc[1585:]

# Verileri Treeview'e ekliyoruz.
for i, row in test_df.iterrows():
    test_data_treeview.insert('', i, values=(i, row['DuyuruIndex'], row['YeniBaslangicTarihi'], row['YeniBaslangicSaati'], row['5HaftaOncesineKadarOrtalamaHiz'], row['Ilce'], row['YolAdresi']))

# Treeview'i grid'e yerleştiriyoruz.
test_data_treeview.grid(row=0, column=0, sticky="wens")

# Seçilen veriyi gösteren bir LabelFrame oluşturuyoruz.
selected_data_frame = ttk.LabelFrame(app, text="Seçilen Veri", padding=(10, 10))
selected_data_frame.grid(row=0, column=1, padx=10, pady=10, sticky="wens")

# Seçilen veriyi gösteren bir Label oluşturuyoruz.
selected_data_label = ttk.Label(selected_data_frame, text="", font=('Helvetica', 14), foreground='#333333')
selected_data_label.grid(row=0, column=0, padx=10, pady=10)

# Tahmin sonuçlarını ve hata oranını gösteren bir LabelFrame oluşturuyoruz.
result_frame = ttk.LabelFrame(app, text="Tahmin Sonuçları", padding=(10, 10))
result_frame.grid(row=1, column=0, columnspan=2, padx=10, pady=10, sticky="wens")

# Tahmin sonuçlarını gösteren bir Label oluşturuyoruz.
result_label = ttk.Label(result_frame, text="", font=('Helvetica', 14), foreground='#333333')
result_label.grid(row=0, column=0, padx=10, pady=10)

# Hata oranını gösteren bir Label oluşturuyoruz.
mape_label = ttk.Label(result_frame, text="", font=('Helvetica', 14), foreground='#333333')
mape_label.grid(row=1, column=0, padx=10, pady=10)

# Filtreleme için giriş widget'ını oluşturuyoruz.
filter_entry = ttk.Entry(test_data_frame, font=('Helvetica', 12), width=10)
filter_entry.grid(row=1, column=0, padx=10, pady=10, sticky="w")

# Filtreleme butonunu oluşturuyoruz.
filter_button = ttk.Button(test_data_frame, text="Filtrele", command=filter_data)
filter_button.grid(row=1, column=0, padx=10, pady=10, sticky="e")

# Grafiği gösteren bir Frame oluşturuyoruz.
graph_frame = ttk.Frame(app)
graph_frame.grid(row=2, column=0, columnspan=2, padx=10, pady=10, sticky="wens")

# Widget'ları büyütüyoruz.
increase_widget_sizes(app)

# Treeview'e seçim yapıldığında on_item_select fonksiyonunu bağlıyoruz.
test_data_treeview.bind('<<TreeviewSelect>>', on_item_select)

# Tkinter uygulamasını başlatıyoruz.
app.mainloop()


In [ ]:
# Train verilerini seç
train_df = beser_dklik_bolumlere_ayrilmis_df.iloc[:1585]
# Test verilerini seç
test_df = beser_dklik_bolumlere_ayrilmis_df.iloc[1585:]

# Feature Selection
featureCols = ['5HaftaOncesineKadarOrtalamaHiz', 'GununBolumu_numeric', 'HaftaninGunu_numeric', 'XCoor', 'YCoor']
x_train = train_df[featureCols]
y_train = train_df['Hiz']

x_test = test_df[featureCols]
y_test = test_df['Hiz']

# Modeli oluşturuyoruz.
model = KNeighborsRegressor()

# Modeli eğitiyoruz.
model.fit(x_train, y_train)

# Test seti üzerinde tahmin yaptırıyoruz.
y_pred = model.predict(x_test)

def mape(y_true, y_pred):
    errors = abs((y_true - y_pred) / y_true)
    mape_value = 100 * np.mean(errors)
    return mape_value

mape_value = mape(y_test, y_pred)
print("MAPE: {:.2f}%".format(mape_value))

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('Gerçek Değerler')
plt.ylabel('Tahmin Edilen Değerler')
plt.title('Gerçek vs Tahmin Edilen Hız Değerleri - KNNR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/KNNRegressor_x_{}_y_Hiz".format(featureCols))

# İngilizce Grafik

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Actual and Predicted Speed Values - KNNR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/KNNRegressor_x_{}_y_Hiz_EN".format(featureCols))

In [ ]:
# Train verilerini seç
train_df = beser_dklik_bolumlere_ayrilmis_df.iloc[:1585]
# Test verilerini seç
test_df = beser_dklik_bolumlere_ayrilmis_df.iloc[1585:]

# Feature Selection
featureCols = ['5HaftaOncesineKadarOrtalamaHiz', 'GununBolumu_numeric', 'HaftaninGunu_numeric', 'XCoor', 'YCoor']
x_train = train_df[featureCols]
y_train = train_df['Hiz']

x_test = test_df[featureCols]
y_test = test_df['Hiz']

# Modeli oluşturuyoruz.
model = DecisionTreeRegressor()

# Modeli eğitiyoruz.
model.fit(x_train, y_train)

# Test seti üzerinde tahmin yaptırıyoruz.
y_pred = model.predict(x_test)

def mape(y_true, y_pred):
    errors = abs((y_true - y_pred) / y_true)
    mape_value = 100 * np.mean(errors)
    return mape_value

mape_value = mape(y_test, y_pred)
print("MAPE: {:.2f}%".format(mape_value))

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('Gerçek Değerler')
plt.ylabel('Tahmin Edilen Değerler')
plt.title('Gerçek vs Tahmin Edilen Hız Değerleri - DTR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/DTRegressor_x_{}_y_Hiz".format(featureCols))

# İngilizce Grafik

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Actual and Predicted Speed Values - DTR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/DTRegressor_x_{}_y_Hiz_EN".format(featureCols))

In [ ]:
# Train verilerini seç
train_df = beser_dklik_bolumlere_ayrilmis_df.iloc[:1585]
# Test verilerini seç
test_df = beser_dklik_bolumlere_ayrilmis_df.iloc[1585:]

# Feature Selection
featureCols = ['5HaftaOncesineKadarOrtalamaHiz', 'GununBolumu_numeric', 'HaftaninGunu_numeric', 'XCoor', 'YCoor']
x_train = train_df[featureCols]
y_train = train_df['Hiz']

x_test = test_df[featureCols]
y_test = test_df['Hiz']

# Modeli oluşturuyoruz.
model = SVR()

# Modeli eğitiyoruz.
model.fit(x_train, y_train)

# Test seti üzerinde tahmin yaptırıyoruz.
y_pred = model.predict(x_test)

def mape(y_true, y_pred):
    errors = abs((y_true - y_pred) / y_true)
    mape_value = 100 * np.mean(errors)
    return mape_value

mape_value = mape(y_test, y_pred)
print("MAPE: {:.2f}%".format(mape_value))

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('Gerçek Değerler')
plt.ylabel('Tahmin Edilen Değerler')
plt.title('Gerçek vs Tahmin Edilen Hız Değerleri - SVR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/SVRegressor_x_{}_y_Hiz".format(featureCols))

# İngilizce Grafik

# Scatter plot oluşturuyoruz.
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

# Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

# Eksen etiketlerini ve başlığı ekliyoruz.
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Actual and Predicted Speed Values - SVR - MAPE: {:.2f}%'.format(mape_value))

# Grafikleri gösteriyoruz.
plt.show()

ax_fig = ax.get_figure()
ax_fig.savefig("Grafikler/SVRegressor_x_{}_y_Hiz_EN".format(featureCols))

In [ ]:
# Günün her bir bölümü için ayrı bir regresyon grafiği çıkaracağız.

parts = ['Sabah', 'Ogle', 'Aksam', 'Gece']
sayac = 0

for i in parts:
    secilen_satirlar = beser_dklik_bolumlere_ayrilmis_df[beser_dklik_bolumlere_ayrilmis_df['GununBolumu'] == i]

    satir = len(secilen_satirlar)

    # Train ve test verilerini istediğimiz yerden ayırıyoruz.
    train_size = int(0.7 * len(secilen_satirlar))
    train_df = secilen_satirlar.iloc[:train_size]
    test_df = secilen_satirlar.iloc[train_size:]
    print("{} icin kullanilacak test veri sayisi:".format(i), len(test_df))

    # Feature Selection
    featureCols = ['5HaftaOncesineKadarOrtalamaHiz', 'HaftaninGunu_numeric', 'XCoor', 'YCoor']
    x_train = train_df[featureCols]
    y_train = train_df['Hiz']

    x_test = test_df[featureCols]
    y_test = test_df['Hiz']

    # Modeli oluşturuyoruz.
    model = RandomForestRegressor(n_estimators=10, random_state = 15)

    # Modeli eğitiyoruz.
    model.fit(x_train, y_train)

    # Test seti üzerinde tahmin yaptırıyoruz.
    y_pred = model.predict(x_test)

    def mape(y_true, y_pred):
        errors = abs((y_true - y_pred) / y_true)
        mape_value = 100 * np.mean(errors)
        return mape_value

    mape_value = mape(y_test, y_pred)
    print("MAPE: {:.2f}%".format(mape_value))

    # Scatter plot oluşturuyoruz.
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

    # Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
    ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

    # Eksen etiketlerini ve başlığı ekliyoruz.
    plt.xlabel('Gerçek Değerler')
    plt.ylabel('Tahmin Edilen Değerler')
    plt.title('{} için Gerçek vs Tahmin Edilen Hız Değerleri - RFR - MAPE: {:.2f}%'.format(i, mape_value))

    # Grafikleri gösteriyoruz.
    plt.show()

    ax_fig = ax.get_figure()
    ax_fig.savefig("Grafikler/{}_Icin_RFRegressor_x_{}_y_Hiz".format(i, featureCols))

    #İngilizce Grafik:

    if sayac == 0:
        segment = 'Morning'
    elif sayac == 1:
        segment = 'Noon'
    elif sayac == 2:
        segment = 'Evening'
    elif sayac == 3:
        segment = 'Night'

    # Scatter plot oluşturuyoruz.
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

    # Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
    ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

    # Eksen etiketlerini ve başlığı ekliyoruz.
    plt.xlabel('True Values %')
    plt.ylabel('Predicted Values %')
    plt.title('Actual and Predicted Speed Values for {} - MAPE: {:.2f} - Row: {}'.format(segment, mape_value, satir))

    # Grafikleri gösteriyoruz.
    plt.show()

    ax_fig = ax.get_figure()
    ax_fig.savefig("Grafikler/RFRegressor_{}_Icin_x_{}_y_Hiz_EN".format(i, featureCols))

    sayac += 1

In [ ]:
# Haftanın her bir günü için ayrı bir regresyon grafiği çıkaracağız.

days = ['Pazartesi', 'Sali', 'Carsamba', 'Persembe', 'Cuma', 'Cumartesi', 'Pazar']
sayac = 0

for i in days:
    secilen_satirlar = beser_dklik_bolumlere_ayrilmis_df[beser_dklik_bolumlere_ayrilmis_df['HaftaninGunu'] == i]

    satir = len(secilen_satirlar)

    # Train ve test verilerini istediğimiz yerden ayırıyoruz.
    train_size = int(0.7 * len(secilen_satirlar))
    train_df = secilen_satirlar.iloc[:train_size]
    test_df = secilen_satirlar.iloc[train_size:]
    print("{} icin kullanilacak test veri sayisi:".format(i), len(test_df))

    # Feature Selection
    featureCols = ['5HaftaOncesineKadarOrtalamaHiz', 'GununBolumu_numeric', 'XCoor', 'YCoor']
    x_train = train_df[featureCols]
    y_train = train_df['Hiz']

    x_test = test_df[featureCols]
    y_test = test_df['Hiz']

    # Modeli oluşturuyoruz.
    model = RandomForestRegressor(n_estimators=10, random_state = 15)

    # Modeli eğitiyoruz.
    model.fit(x_train, y_train)

    # Test seti üzerinde tahmin yaptırıyoruz.
    y_pred = model.predict(x_test)

    def mape(y_true, y_pred):
        errors = abs((y_true - y_pred) / y_true)
        mape_value = 100 * np.mean(errors)
        return mape_value

    mape_value = mape(y_test, y_pred)
    print("MAPE: {:.2f}%".format(mape_value))

    # Scatter plot oluşturuyoruz.
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

    # Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
    ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

    # Eksen etiketlerini ve başlığı ekliyoruz.
    plt.xlabel('Gerçek Değerler')
    plt.ylabel('Tahmin Edilen Değerler')
    plt.title('{} için Gerçek vs Tahmin Edilen Hız Değerleri - RFR - MAPE: {:.2f}%'.format(i, mape_value))

    # Grafikleri gösteriyoruz.
    plt.show()

    ax_fig = ax.get_figure()
    ax_fig.savefig("Grafikler/{}_Icin_RFRegressor_x_{}_y_Hiz".format(i, featureCols))

    # İngilizce Grafik:

    if sayac == 0:
        day = 'Monday'
    elif sayac == 1:
        day = 'Tuesday'
    elif sayac == 2:
        day = 'Wednesday'
    elif sayac == 3:
        day = 'Thursday'
    elif sayac == 4:
        day = 'Friday'
    elif sayac == 5:
        day = 'Saturday'
    elif sayac == 6:
        day = 'Sunday'

    # Scatter plot oluşturuyoruz.
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

    # Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.___
    ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

    # Eksen etiketlerini ve başlığı ekliyoruz.
    plt.xlabel('True Values %')
    plt.ylabel('Predicted Values %')
    plt.title('Actual and Predicted Speed Values for {} - MAPE: {:.2f}% - Row: {}'.format(day, mape_value, satir))

    # Grafikleri gösteriyoruz.
    plt.show()

    ax_fig = ax.get_figure()
    ax_fig.savefig("Grafikler/RFRegressor_{}_Icin_x_{}_y_Hiz_EN".format(i, featureCols))

    sayac += 1

In [ ]:
# Haftanın her bir günü için ayrı bir regresyon grafiği çıkaracağız.

ilceler = ['Beykoz', 'Sultangazi', 'Üsküdar', 'Esenyurt', 'Bağcılar',
       'Bahçelievler', 'Eyüpsultan', 'Çatalca', 'Ataşehir',
       'Küçükçekmece', 'Ümraniye', 'Kâğıthane', 'Kadıköy', 'Sarıyer',
       'Şişli', 'Fatih', 'Beyoğlu', 'Tuzla', 'Çekmeköy', 'Büyükçekmece',
       'Avcılar', 'Pendik', 'Maltepe', 'Sultanbeyli', 'Bayrampaşa',
       'Esenler', 'Silivri', 'Başakşehir', 'Zeytinburnu', 'Beylikdüzü']

for i in ilceler:
    secilen_satirlar = beser_dklik_bolumlere_ayrilmis_df[beser_dklik_bolumlere_ayrilmis_df['Ilce'] == i]

    satir = len(secilen_satirlar)
    
    if not(satir > 200):
        continue

    # Train ve test verilerini istediğimiz yerden ayırıyoruz.
    train_size = int(0.7 * len(secilen_satirlar))
    train_df = secilen_satirlar.iloc[:train_size]
    test_df = secilen_satirlar.iloc[train_size:]
    print("{} icin kullanilacak test veri sayisi:".format(i), len(test_df))

    # Feature Selection
    featureCols = ['5HaftaOncesineKadarOrtalamaHiz', 'GununBolumu_numeric', 'HaftaninGunu_numeric', 'XCoor', 'YCoor']
    x_train = train_df[featureCols]
    y_train = train_df['Hiz']

    x_test = test_df[featureCols]
    y_test = test_df['Hiz']

    # Modeli oluşturuyoruz.
    model = RandomForestRegressor(n_estimators=10, random_state = 15)

    # Modeli eğitiyoruz.
    model.fit(x_train, y_train)

    # Test seti üzerinde tahmin yaptırıyoruz.
    y_pred = model.predict(x_test)

    def mape(y_true, y_pred):
        errors = abs((y_true - y_pred) / y_true)
        mape_value = 100 * np.mean(errors)
        return mape_value

    mape_value = mape(y_test, y_pred)
    print("MAPE: {:.2f}%".format(mape_value))

    # Scatter plot oluşturuyoruz.
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

    # Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
    ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

    # Eksen etiketlerini ve başlığı ekliyoruz.
    plt.xlabel('Gerçek Değerler %')
    plt.ylabel('Tahmin Edilen Değerler %')
    plt.title('{} için Gerçek vs Tahmin Edilen Hız Değerleri - RFR - MAPE: {:.2f}%'.format(i, mape_value))

    # Grafikleri gösteriyoruz.
    plt.show()

    ax_fig = ax.get_figure()
    ax_fig.savefig("Grafikler/{}_Icin_RFRegressor_x_{}_y_Hiz".format(i, featureCols))

    # İngilizce Grafik:

    # Scatter plot oluşturuyoruz.
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_test, y=y_pred, color='blue', alpha=0.5)

    # Gerçek ve tahmin edilen değerler arasında bir doğru çizdiriyoruz.
    ax = sns.lineplot(x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)], color='red', linestyle='--')

    # Eksen etiketlerini ve başlığı ekliyoruz.
    plt.xlabel('True Values %')
    plt.ylabel('Predicted Values %')
    plt.title('Actual and Predicted Speed Values for {} - MAPE: {:.2f} - Row: {}'.format(i, mape_value, satir))

    # Grafikleri gösteriyoruz.
    plt.show()

    ax_fig = ax.get_figure()
    ax_fig.savefig("Grafikler/RFRegressor_{}_Icin_x_{}_y_Hiz_EN".format(i, featureCols))